# 《Back to Basics: Let Denoising Generative Models Denoise》论文详解

**论文作者：Tianhong Li、Kaiming He（MIT）**  
**论文版本：arXiv:2511.13720v2，2026年1月7日**  
**论文地址：** https://arxiv.org/pdf/2511.13720

这次我以论文 v2 原文为依据，按照作者的章节顺序进行讲解。对于关键数学结论，会补充原文省略的中间推导，并区分**论文实际证明或实验验证的结论**与**为了帮助理解而补充的数学解释**。

全文结构如下：

| 原文章节 | 主要内容 |
|---|---|
| Abstract / 1. Introduction | 核心观点、研究动机 |
| 2. Related Work | 扩散模型、去噪模型、流形学习与像素空间生成 |
| 3. On Prediction Outputs of Diffusion Models | 预测空间、损失空间、生成空间及数学关系 |
| 3.3 Toy Experiment | 验证低维流形假设与高维预测困难 |
| 4. Just Image Transformers | JiT 架构、消融实验、训练与采样算法 |
| 5. Comparisons | 高分辨率生成、模型扩展性及性能对比 |
| 6. Discussion and Conclusion | 研究结论与意义 |
| Appendix A–C | 实现细节、额外实验、预条件化分析等 |

---

# 第一部分：全文核心观点

## 1. 论文到底研究什么问题？

扩散模型最初的思想是去噪：

给定一个被噪声污染的数据：

$$
z_t
$$

训练神经网络恢复真实数据：

$$
x
$$

但是随着扩散模型的发展，主流方法逐渐不再让神经网络直接预测干净数据。

例如，DDPM 常采用噪声预测：

$$
\epsilon_\theta(z_t,t)
$$

Flow Matching 则通常采用速度预测：

$$
v_\theta(z_t,t)
$$

而实际上，这三种预测方式在数学上可以相互转换：

$$
\boxed{
x\longleftrightarrow\epsilon\longleftrightarrow v
}
\tag{1.1}
$$

因此，一个长期存在的观点是：

只要调整对应的损失权重，预测 $x$、$\epsilon$ 或 $v$ 应该没有本质区别。

**这篇论文挑战的正是这一认识。**

作者指出：

> 数学上能够相互转换，并不代表有限容量神经网络学习这些目标时具有相同的难度。

论文进一步认为，真实数据与噪声之间存在一个重要的几何差异。

真实数据往往满足低维流形假设，而噪声及包含噪声的速度不满足这一结构。

因此：

$$
\boxed{
\text{直接预测干净数据，可能比直接预测高维噪声或速度更容易}
}
\tag{1.2}
$$

这一差异在高维数据空间中尤其明显。[arXiv](https://arxiv.org/html/2511.13720?utm_source=chatgpt.com)

## 2. 全文的五个核心结论

### 结论一：Prediction Space 与 Loss Space 应当分开考虑

论文将两个概念明确区分。

**Prediction Space（预测空间）**决定神经网络直接输出什么：

$$
\operatorname{Net}_\theta(z_t,t)
\rightarrow
\begin{cases}
\hat x\\
\hat\epsilon\\
\hat v
\end{cases}
$$

**Loss Space（损失空间）**决定在哪个变量空间计算 MSE：

$$
L_x,\qquad L_\epsilon,\qquad L_v
$$

两者可以自由组合，共有九种基本配置。

例如，完全可以：

$$
\operatorname{Net}_\theta(z_t,t)=\hat x
$$

但训练时计算：

$$
L_v=\|\hat v-v\|^2
$$

论文最终采用的正是：

$$
\boxed{
x\text{-prediction}+v\text{-loss}
}
\tag{1.3}
$$

### 结论二：真实数据与噪声具有不同的几何结构

假设真实数据：

$$
x\in\mathbb R^D
$$

其中 $D$ 是环境维度（Ambient Dimension）。

根据低维流形假设，真实数据集中在某个流形：

$$
\mathcal M\subset\mathbb R^D
$$

上，其内在维度：

$$
d=\dim(\mathcal M)
$$

满足：

$$
\boxed{d\ll D}
\tag{1.4}
$$

也就是说，虽然真实数据具有 $D$ 个坐标，但实际自由度可能远小于 $D$。

相比之下，高斯噪声：

$$
\epsilon\sim\mathcal N(0,I_D)
$$

具有全维分布。

Flow Matching 的速度：

$$
v=x-\epsilon
$$

也包含全维噪声信息。

因此，直接预测 $x$ 可以利用数据本身的低维结构，而直接预测 $\epsilon$ 或 $v$ 可能需要网络保留更多高维信息。

### 结论三：高维数据下，预测空间比损失空间更关键

这是论文最重要的实验发现之一。

作者固定相同的网络架构，比较九种 Prediction Space 与 Loss Space 组合。

在 ImageNet 256×256、patch size 为16的实验中：

| Loss Space | x-pred | ε-pred | v-pred |
|---|---:|---:|---:|
| x-loss | 10.14 | 379.21 | 107.55 |
| ε-loss | 10.45 | 394.58 | 126.88 |
| v-loss | **8.62** | 372.38 | 96.53 |

评价指标是 FID，越低越好。

可以发现：

- 只要直接预测 $x$，三种 Loss Space 都能正常训练。
- 如果直接预测 $\epsilon$ 或 $v$，即使改变 Loss Space，也无法解决严重的性能退化。
- 在 $x$-prediction 内部，v-loss 的表现最好。

因此：

$$
\boxed{
\text{高维生成中的困难，不能仅通过调整 Loss Weighting 解决}
}
\tag{1.5}
$$

这是论文用实验支持的结论，并不是所有模型和数据集上的普遍定理。[arXiv](https://arxiv.org/html/2511.13720?utm_source=chatgpt.com)

### 结论四：网络隐藏维度不一定需要大于数据维度

传统直觉认为：

如果输入或输出维度为：

$$
D=3072
$$

那么网络隐藏维度最好足够大，避免严重的信息瓶颈。

但作者发现，使用 $x$-prediction 时，即使：

$$
H\ll D
$$

模型仍然能够正常生成图像。

更有意思的是，作者主动引入低维 Bottleneck，反而可以改善生成质量。

这说明：

**对于具有低维结构的真实数据，网络未必需要在隐藏表示中完整保留高维观测空间的所有信息。**

### 结论五：可以不使用 VAE，直接在原始像素空间训练 Transformer

作者提出：

**JiT：Just image Transformers**

其主要特点是：

- 不使用 VAE 或其他图像 tokenizer。
- 不依赖预训练模型。
- 不使用感知损失、对抗损失或表示对齐损失。
- 使用普通 Vision Transformer 直接处理原始像素。
- 网络直接预测干净图像。
- 采用 Flow Matching 的速度损失训练。
- 使用 ODE 求解器生成图像。

论文在 ImageNet 256×256 和 512×512 上获得了具有竞争力的结果。

这意味着：

$$
\boxed{
\text{Diffusion}+\text{Transformer}
}
$$

可以直接在原始高维数据空间中工作，而不一定需要预先训练一个 VAE 来降低维度。

---

# 第二部分：按照原文顺序详解

# 1. Introduction：研究动机

## 1.1 扩散模型为什么逐渐不再直接预测干净数据？

论文首先回顾扩散模型的发展。

早期扩散模型的目标可以理解为：

$$
\text{Noisy Data}\rightarrow\text{Clean Data}
$$

也就是经典的去噪问题。

但是 DDPM 的成功，使得噪声预测逐渐成为主流。

假设前向加噪过程：

$$
z_t=\alpha_t x+\sigma_t\epsilon
\tag{1.6}
$$

其中：

- $x$：干净数据。
- $\epsilon\sim\mathcal N(0,I)$：标准高斯噪声。
- $\alpha_t$：数据系数。
- $\sigma_t$：噪声系数。
- $z_t$：加噪后的数据。

DDPM 通常训练：

$$
\epsilon_\theta(z_t,t)\approx\epsilon
$$

对应：

$$
L_\epsilon
=
\mathbb E\left[
\|\epsilon_\theta(z_t,t)-\epsilon\|^2
\right]
\tag{1.7}
$$

后来，Flow Matching 使用速度预测：

$$
v_\theta(z_t,t)\approx v
$$

这两种方式的共同点是：

**网络直接输出的不是干净数据，而是包含高维噪声信息的量。**

作者认为，这种选择在低维空间中可能没有明显问题，但在原始像素空间等高维任务中，可能导致神经网络的表示压力显著增加。

## 1.2 为什么过去没有发现这个问题？

作者给出两个重要原因。

第一，很多早期扩散模型使用 U-Net。

U-Net 具有大量卷积通道和跨层 Skip Connection，可以在一定程度上保留高维输入信息。

例如：

$$
H\times W\times3
$$

的图像经过卷积后，可以得到：

$$
H\times W\times128
$$

的高维特征。

第二，现代扩散模型广泛使用 Latent Diffusion。

原始图像首先经过 VAE：

$$
x_{\text{pixel}}
\xrightarrow{\text{VAE Encoder}}
z_{\text{latent}}
$$

然后在低维潜在空间训练扩散模型。

这相当于提前缓解了高维信息表示的问题。

但是，如果直接使用普通 ViT 处理原始像素：

$$
x_{\text{pixel}}
\xrightarrow{\text{Patch Embedding}}
h
$$

就可能出现：

$$
D_{\text{patch}}>H_{\text{hidden}}
$$

此时，网络必须将高维输入压缩到较低维的隐藏表示。

如果直接预测高维噪声，信息瓶颈就可能成为严重问题。

作者由此提出：

**与其不断增加模型宽度、减小 patch 或引入额外模块，不如重新考虑网络究竟应该预测什么。**

---

# 2. Related Work：相关研究

这一节主要解释为什么作者的研究虽然使用了经典的 $x$-prediction，却仍然具有新的意义。

## 2.1 Diffusion Models and Their Predictions

论文回顾三类常见预测目标：

$$
x,\quad\epsilon,\quad v
$$

DDPM 推广了 $\epsilon$-prediction。

后续研究引入了 $v$-prediction，并分析不同预测目标之间的转换关系。

EDM 则将扩散模型重新组织为去噪器形式。

这里有一个需要特别注意的区别。

EDM 虽然定义：

$$
D_\theta(z,\sigma)
$$

作为去噪器，但它的神经网络直接输出通常经过预条件化组合：

$$
D_\theta(z,\sigma)
=
c_{\text{skip}}(\sigma)z
+
c_{\text{out}}(\sigma)
F_\theta(c_{\text{in}}(\sigma)z,c_{\text{noise}}(\sigma))
\tag{2.1}
$$

其中：

- $F_\theta$：神经网络直接输出。
- $c_{\text{skip}}$：输入残差系数。
- $c_{\text{out}}$：网络输出缩放系数。
- $c_{\text{in}}$：输入缩放系数。
- $c_{\text{noise}}$：噪声条件变换。

因此：

$$
F_\theta\neq D_\theta
$$

也就是说：

**最终模型是一个去噪器，不代表神经网络本身直接预测干净数据。**

这是作者与 EDM 区分的重要地方。

## 2.2 Denoising Models

作者将研究联系到传统去噪方法和 Denoising Autoencoder（DAE）。

传统去噪问题是：

$$
z=x+\sigma\epsilon
$$

训练：

$$
D_\theta(z)\approx x
$$

其目标为：

$$
L=\mathbb E\|D_\theta(z)-x\|^2
\tag{2.2}
$$

为什么这种训练可以学习有意义的数据表示？

因为真实数据往往存在大量相关性。

例如自然图像中，相邻像素、纹理、边缘和物体结构并不是独立的。

去噪器需要识别这些相关结构，才能从噪声中恢复数据。

因此，去噪任务本身可以促使模型学习数据流形。

作者认为，现代扩散模型可以重新利用这一思想。

## 2.3 Manifold Learning

这是论文最重要的理论背景。

### 2.3.1 什么是低维流形？

假设数据：

$$
x\in\mathbb R^D
$$

但实际上可以由少量参数决定：

$$
x=g(u)
$$

其中：

$$
u\in\mathbb R^d,\qquad d\ll D
$$

如果映射 $g$ 足够光滑，并且局部具有满秩雅可比矩阵，那么它可以在局部描述一个 $d$ 维流形。

例如，固定半径的圆：

$$
x_1=r\cos\theta
$$

$$
x_2=r\sin\theta
$$

虽然：

$$
(x_1,x_2)\in\mathbb R^2
$$

但是只需要：

$$
\theta\in\mathbb R
$$

这一个局部自由度。

所以圆是嵌入二维空间中的一维流形。

对于自然图像也是类似的思想。

虽然图片可能具有数十万甚至数百万个像素值，但真实图像受到物体形状、材质、光照、透视关系等因素约束。

因此，数据并不会任意填满整个像素空间。

### 2.3.2 为什么 Bottleneck 可能有利？

假设：

$$
x=g(u),\qquad u\in\mathbb R^d
$$

一个网络可以学习：

$$
z
\xrightarrow{\text{Encoder}}
h
\xrightarrow{\text{Decoder}}
\hat x
$$

其中：

$$
h\in\mathbb R^H
$$

如果：

$$
d\leq H<D
$$

那么隐藏表示虽然无法无损编码任意 $D$ 维输入，但仍可能足以表示数据流形上的有效信息。

这就是作者研究 Bottleneck 的理论动机。

需要强调，低维流形并不保证任意网络都能轻松学习它；流形的曲率、拓扑、数据分布和网络结构也会影响难度。

## 2.4 Pixel-space Diffusion

作者指出，直接在像素空间生成图像并不是新问题。

早期 DDPM、ADM 等方法就使用像素空间扩散。

但这些模型往往依赖卷积 U-Net。

对于普通 ViT：

$$
D_{\text{patch}}=p^2C
\tag{2.3}
$$

其中：

- $p$：patch 边长。
- $C$：图像通道数。
- $D_{\text{patch}}$：每个 patch 展平后的维度。

例如 RGB 图像，patch size 为16：

$$
D_{\text{patch}}=16^2\times3=768
$$

如果 patch size 为32：

$$
D_{\text{patch}}=32^2\times3=3072
$$

而 Transformer hidden dimension 可能只有768。

这意味着：

$$
3072\rightarrow768
$$

存在明显的信息压缩。

作者认为，直接预测噪声或速度时，这种压缩可能成为主要障碍。

## 2.5 High-dimensional Diffusion

作者回顾了高维扩散生成中出现的性能退化问题。

以往的一些解决方案包括：

- 使用更小的 patch。
- 增大隐藏维度。
- 使用多尺度网络。
- 增加跳跃连接。
- 使用自监督预训练或额外的表示学习目标。

作者希望证明：

**高维生成的困难不一定需要通过更复杂的架构解决，改变 Prediction Space 本身就可能产生显著改善。**

## 2.6 x-prediction

作者明确承认：

$x$-prediction 并不是新方法。

它在早期扩散模型中就已经存在。

论文的贡献不是发明新的预测目标，而是重新解释：

**为什么直接预测干净数据，在高维空间中可能比预测噪声和速度更合理。**

这也为后面的第三节奠定了基础。[arXiv](https://arxiv.org/html/2511.13720?utm_source=chatgpt.com)

---

# 3. On Prediction Outputs of Diffusion Models

这一节是整篇论文的数学核心。

作者在这里正式定义：

1. 扩散与 Flow Matching 的统一表示。
2. Prediction Space 和 Loss Space 的区别。
3. 三种预测目标之间的转换关系。
4. 为什么不同预测目标在数学上等价，但在神经网络中不等价。
5. 用 Toy Experiment 验证高维表示瓶颈问题。

下面从论文公式（1）开始推导。

## 3.1 Background: Diffusion and Flows

### 3.1.1 统一数据与噪声的定义

按照你之前学习 Flow Matching 的习惯，这里采用：

$$
\boxed{
x_0=\text{初始噪声},\qquad x_1=\text{真实数据}
}
$$

与论文符号的对应关系为：

| 论文符号 | 本文讲解符号 | 含义 |
|---|---|---|
| $\epsilon$ | $x_0$ | 初始高斯噪声 |
| $x$ | $x_1$ | 真实数据 |
| $z_t$ | $x_t$ | 中间状态 |
| $v$ | $u$ | 速度 |

这里把速度记为 $u$，避免与时间下标混淆。

设：

$$
x_0\sim\mathcal N(0,I_D)
$$

$$
x_1\sim p_{\mathrm{data}}
$$

其中：

$$
x_0,x_1\in\mathbb R^D
$$

论文采用线性插值：

$$
\boxed{
x_t=tx_1+(1-t)x_0
}
\tag{3.1}
$$

时间范围：

$$
t\in[0,1]
$$

因此：

$$
t=0\Rightarrow x_t=x_0
$$

$$
t=1\Rightarrow x_t=x_1
$$

这与 Conditional Flow Matching 的标准 OT 直线路径形式相同。

需要注意，论文在这里使用的是独立采样的数据与噪声端点，并没有要求计算 minibatch OT coupling。

### 3.1.2 推导真实速度

速度定义为：

$$
u_t=\frac{dx_t}{dt}
\tag{3.2}
$$

将式（3.1）代入：

$$
u_t
=
\frac{d}{dt}
\left[
tx_1+(1-t)x_0
\right]
$$

由于训练时固定一组：

$$
(x_0,x_1)
$$

所以：

$$
\frac{dx_0}{dt}=0,\qquad
\frac{dx_1}{dt}=0
$$

得到：

$$
u_t=x_1-x_0
$$

因此：

$$
\boxed{
u=x_1-x_0
}
\tag{3.3}
$$

对于固定端点的条件路径，速度与时间无关。

但这里要区分：

**单条条件路径的速度恒定，不代表模型学习到的边际速度场也恒定。**

因为模型只观察到：

$$
(x_t,t)
$$

并不知道生成当前 $x_t$ 的具体端点。

对于平方误差，理想的边际速度场实际上是：

$$
\boxed{
u^*(x,t)
=
\mathbb E[x_1-x_0\mid x_t=x]
}
\tag{3.4}
$$

这一点可以通过条件期望的最小二乘性质推导。

假设模型预测：

$$
f(x_t,t)
$$

训练目标：

$$
L(f)=
\mathbb E\|f(x_t,t)-u\|^2
$$

令：

$$
m(x_t,t)=\mathbb E[u\mid x_t,t]
$$

则：

$$
f-u=(f-m)+(m-u)
$$

展开平方：

$$
\|f-u\|^2
=
\|f-m\|^2
+
\|m-u\|^2
+
2(f-m)^\top(m-u)
$$

对给定 $x_t,t$ 取条件期望。

由于：

$$
\mathbb E[m-u\mid x_t,t]=0
$$

交叉项消失。

于是：

$$
L(f)
=
\mathbb E\|f-m\|^2
+
\mathbb E\|m-u\|^2
$$

第二项与模型无关。

因此最优解为：

$$
\boxed{
f^*(x_t,t)
=
\mathbb E[u\mid x_t,t]
}
\tag{3.5}
$$

这是对论文 Flow Matching 背景公式的补充推导。

### 3.1.3 Flow Matching 的训练目标

标准训练目标为：

$$
\boxed{
L_u
=
\mathbb E_{t,x_0,x_1}
\left[
\|u_\theta(x_t,t)-(x_1-x_0)\|^2
\right]
}
\tag{3.6}
$$

其中：

$$
u_\theta(x_t,t)
$$

是模型构造出的速度预测。

注意这里没有要求速度一定是网络的直接输出。

这一点正是后面 Prediction Space 与 Loss Space 解耦的基础。

### 3.1.4 推理过程

模型训练完成后，求解：

$$
\boxed{
\frac{dx_t}{dt}=u_\theta(x_t,t)
}
\tag{3.7}
$$

初始状态：

$$
x_0\sim\mathcal N(0,I)
$$

最终得到：

$$
x_1^{\mathrm{gen}}
$$

例如 Euler 积分：

$$
\boxed{
x_{t+\Delta t}
=
x_t+\Delta t\,u_\theta(x_t,t)
}
\tag{3.8}
$$

论文默认采用50步 Heun 求解器，而不是必须使用 Euler。

Heun 的更新为：

首先预测当前速度：

$$
k_1=u_\theta(x_t,t)
$$

使用 Euler 预测下一状态：

$$
\tilde x_{t+\Delta t}=x_t+\Delta t k_1
$$

再预测下一时刻速度：

$$
k_2=u_\theta(\tilde x_{t+\Delta t},t+\Delta t)
$$

最终：

$$
\boxed{
x_{t+\Delta t}
=
x_t+\frac{\Delta t}{2}(k_1+k_2)
}
\tag{3.9}
$$

Heun 使用两个速度估计的平均值，提高积分精度。

---

## 3.2 Prediction Space and Loss Space

这是论文最重要的一节。

作者强调：

**模型的直接输出、训练损失所在空间、最终用于采样的速度场，是三个可以区分的概念。**

### 3.2.1 首先推导三个变量之间的转换关系

已知：

$$
x_t=tx_1+(1-t)x_0
\tag{3.10}
$$

$$
u=x_1-x_0
\tag{3.11}
$$

我们希望证明，只要知道：

$$
x_t,t
$$

以及：

$$
x_1,\quad x_0,\quad u
$$

中的任意一个，就可以恢复另外两个。

#### 情况一：已知真实数据 $x_1$

由：

$$
x_t=tx_1+(1-t)x_0
$$

移项：

$$
(1-t)x_0=x_t-tx_1
$$

得到：

$$
\boxed{
x_0=\frac{x_t-tx_1}{1-t}
}
\tag{3.12}
$$

又因为：

$$
u=x_1-x_0
$$

代入：

$$
u=x_1-\frac{x_t-tx_1}{1-t}
$$

通分：

$$
u=
\frac{(1-t)x_1-x_t+tx_1}{1-t}
$$

整理：

$$
\boxed{
u=\frac{x_1-x_t}{1-t}
}
\tag{3.13}
$$

所以：

$$
\boxed{
x_1
\Rightarrow
\begin{cases}
x_0=\dfrac{x_t-tx_1}{1-t}\\[6pt]
u=\dfrac{x_1-x_t}{1-t}
\end{cases}
}
\tag{3.14}
$$

#### 情况二：已知噪声 $x_0$

同样从：

$$
x_t=tx_1+(1-t)x_0
$$

得到：

$$
tx_1=x_t-(1-t)x_0
$$

因此：

$$
\boxed{
x_1=\frac{x_t-(1-t)x_0}{t}
}
\tag{3.15}
$$

速度：

$$
u=x_1-x_0
$$

代入：

$$
u=
\frac{x_t-(1-t)x_0}{t}-x_0
$$

整理：

$$
\boxed{
u=\frac{x_t-x_0}{t}
}
\tag{3.16}
$$

因此：

$$
\boxed{
x_0
\Rightarrow
\begin{cases}
x_1=\dfrac{x_t-(1-t)x_0}{t}\\[6pt]
u=\dfrac{x_t-x_0}{t}
\end{cases}
}
\tag{3.17}
$$

#### 情况三：已知速度 $u$

由：

$$
u=x_1-x_0
$$

得到：

$$
x_1=x_0+u
$$

代入插值公式：

$$
x_t=t(x_0+u)+(1-t)x_0
$$

展开：

$$
x_t=tx_0+tu+x_0-tx_0
$$

因此：

$$
x_t=x_0+tu
$$

得到：

$$
\boxed{
x_0=x_t-tu
}
\tag{3.18}
$$

再由：

$$
x_1=x_0+u
$$

得到：

$$
x_1=x_t-tu+u
$$

因此：

$$
\boxed{
x_1=x_t+(1-t)u
}
\tag{3.19}
$$

所以：

$$
\boxed{
u
\Rightarrow
\begin{cases}
x_0=x_t-tu\\
x_1=x_t+(1-t)u
\end{cases}
}
\tag{3.20}
$$

### 3.2.2 Prediction Space：模型直接预测什么？

根据上面的推导，可以选择三种模型参数化。

**第一种：直接预测真实数据**

$$
\boxed{
\hat x_1=f_\theta(x_t,t)
}
\tag{3.21}
$$

然后：

$$
\hat u=\frac{\hat x_1-x_t}{1-t}
$$

**第二种：直接预测噪声**

$$
\boxed{
\hat x_0=f_\theta(x_t,t)
}
\tag{3.22}
$$

然后：

$$
\hat u=\frac{x_t-\hat x_0}{t}
$$

**第三种：直接预测速度**

$$
\boxed{
\hat u=f_\theta(x_t,t)
}
\tag{3.23}
$$

然后：

$$
\hat x_1=x_t+(1-t)\hat u
$$

虽然三者可以转换，但它们对应的网络函数形式不同。

例如：

$$
\hat u=f_\theta(x_t,t)
$$

与：

$$
\hat u=\frac{f_\theta(x_t,t)-x_t}{1-t}
$$

不是同一个函数族，除非网络架构有足够能力实现相应变换。

这就引出了论文的核心问题：

**数学上等价的输出参数化，是否会受到神经网络容量的限制？**

作者认为会，并在后面通过实验验证。

---

### 3.2.3 Loss Space：损失在哪个空间计算？

Loss Space 可以选择：

$$
L_{x_1}
=
\mathbb E\|\hat x_1-x_1\|^2
\tag{3.24}
$$

$$
L_{x_0}
=
\mathbb E\|\hat x_0-x_0\|^2
\tag{3.25}
$$

$$
L_u
=
\mathbb E\|\hat u-u\|^2
\tag{3.26}
$$

因此有：

$$
3\times3=9
$$

种组合。

下面推导它们之间的数学关系。

### 3.2.4 重点推导：真实数据预测 + 三种 Loss

假设网络直接预测：

$$
\hat x_1=f_\theta(x_t,t)
$$

定义预测误差：

$$
e=\hat x_1-x_1
\tag{3.27}
$$

**第一种：数据空间损失**

直接：

$$
\boxed{
L_{x_1}=\mathbb E\|e\|^2
}
\tag{3.28}
$$

**第二种：噪声空间损失**

根据：

$$
\hat x_0=\frac{x_t-t\hat x_1}{1-t}
$$

真实噪声：

$$
x_0=\frac{x_t-tx_1}{1-t}
$$

两式相减：

$$
\hat x_0-x_0
=
\frac{x_t-t\hat x_1-x_t+tx_1}{1-t}
$$

得到：

$$
\hat x_0-x_0
=
-\frac{t}{1-t}(\hat x_1-x_1)
$$

因此：

$$
\boxed{
L_{x_0}
=
\mathbb E\left[
\frac{t^2}{(1-t)^2}\|e\|^2
\right]
}
\tag{3.29}
$$

**第三种：速度空间损失**

预测速度：

$$
\hat u=\frac{\hat x_1-x_t}{1-t}
$$

真实速度：

$$
u=\frac{x_1-x_t}{1-t}
$$

相减：

$$
\hat u-u
=
\frac{\hat x_1-x_1}{1-t}
$$

所以：

$$
\boxed{
L_u
=
\mathbb E\left[
\frac{1}{(1-t)^2}\|e\|^2
\right]
}
\tag{3.30}
$$

这就是论文选择：

$$
x\text{-prediction}+v\text{-loss}
$$

时最重要的数学关系。

### 3.2.5 完整的九种组合

为了方便比较，定义：

$$
e_x=\hat x_1-x_1
$$

$$
e_\epsilon=\hat x_0-x_0
$$

$$
e_u=\hat u-u
$$

则九种组合对应：

| Prediction Space | 数据 Loss | 噪声 Loss | 速度 Loss |
|---|---|---|---|
| $x_1$ | $\|e_x\|^2$ | $\frac{t^2}{(1-t)^2}\|e_x\|^2$ | $\frac{1}{(1-t)^2}\|e_x\|^2$ |
| $x_0$ | $\frac{(1-t)^2}{t^2}\|e_\epsilon\|^2$ | $\|e_\epsilon\|^2$ | $\frac1{t^2}\|e_\epsilon\|^2$ |
| $u$ | $(1-t)^2\|e_u\|^2$ | $t^2\|e_u\|^2$ | $\|e_u\|^2$ |

表中每个表达式都位于对 $t,x_0,x_1$ 取期望的损失内部。

这个表格对应论文 Table 1 的数学内容。

### 3.2.6 从这些公式能得到什么结论？

首先，固定 Prediction Space 时，不同 Loss Space 相当于不同的时间权重。

例如固定真实数据预测：

$$
\hat x_1=f_\theta(x_t,t)
$$

那么：

$$
L=\mathbb E_t[w(t)\|e_x\|^2]
$$

其中：

$$
w(t)=
\begin{cases}
1,&\text{数据 Loss}\\[3pt]
\dfrac{t^2}{(1-t)^2},&\text{噪声 Loss}\\[8pt]
\dfrac{1}{(1-t)^2},&\text{速度 Loss}
\end{cases}
\tag{3.31}
$$

因此，Loss Space 影响不同时间步的监督强度。

但是：

**Prediction Space 改变的并不只是损失权重，而是网络需要直接表示的目标函数。**

这就是论文强调二者必须区分的原因。

还有一个数学细节：当 $t\to1$ 时，式（3.30）的权重会发散。因此论文实际实现时对分母进行了截断，默认最小值为0.05。截断后的损失不再严格等于未经截断的式（3.30）。

---

### 3.2.7 Generator Space：最终采样空间

作者还区分了第三个概念：

**Generator Space（生成空间）**。

它指最终如何使用模型输出构造生成过程。

例如，无论网络直接预测什么，最后都可以恢复速度：

$$
\hat u(x_t,t)
$$

然后求解：

$$
\frac{dx_t}{dt}=\hat u(x_t,t)
$$

因此：

$$
\boxed{
\text{Prediction Space}\neq\text{Loss Space}\neq\text{Generator Space}
}
\tag{3.32}
$$

三者关注的分别是：

- 网络直接学习什么。
- 训练误差如何计算。
- 推理时如何生成样本。

对于这篇论文，最终选择：

$$
\boxed{
\begin{aligned}
\text{Prediction Space}&:x_1\\
\text{Loss Space}&:u\\
\text{Generator Space}&:\text{ODE}
\end{aligned}
}
\tag{3.33}
$$

---

## 3.3 Toy Experiment：低维子空间与高维噪声能量

这一节讨论论文的 Toy Experiment。正文直接使用投影矩阵和各向同性噪声的基础结论；这些结论的完整证明统一放在全文末尾的**附录 C**。

先固定符号：

$$
\boxed{
\begin{aligned}
x_0&:\text{初始高斯噪声},\\
x_1&:\text{真实数据},\\
x_t&:\text{时间 }t\text{ 的插值状态},\\
v&:\text{从噪声指向真实数据的条件速度}.
\end{aligned}
}
\tag{3.34}
$$

其中：

$$
x_t=(1-t)x_0+tx_1,qquad v=x_1-x_0.
\tag{3.35}
$$

这里的 $v$ 是由一对端点 $(x_0,x_1)$ 构造的**条件速度目标**，并不等同于对所有可能端点取条件期望后得到的边际速度场。

### 3.3.1 实验构造：把二维数据嵌入 $D$ 维空间

先在低维空间采样真实数据：

$$
z_1\in\mathbb R^d,qquad d=2.
$$

再通过列正交矩阵 $P$ 嵌入环境空间：

$$
\boxed{x_1=Pz_1},qquad
P\in\mathbb R^{D\times d},qquad P^\top P=I_d.
\tag{3.36}
$$

因此 $x_1\in\mathbb R^D$，但所有真实数据都位于同一个 $d$ 维线性子空间：

$$
\mathcal S=\operatorname{Col}(P),qquad \dim(\mathcal S)=d.
\tag{3.37}
$$

论文固定 $d=2$，改变环境维度：

$$
D\in\{2,8,16,512\}.
$$

所以环境坐标可以越来越多，但数据的内在自由度始终只有两个。严格地说，这里构造的是低维**线性子空间**中的分布，它是低维流形的一个特殊情形。

### 3.3.2 正文使用的数学基础结论

令 $\Pi$ 表示到数据子空间 $\mathcal S$ 的正交投影，$\Pi_\perp$ 表示到正交补空间的投影。由于 $P$ 的列标准正交：

$$
\boxed{\Pi=PP^\top},qquad
\boxed{\Pi_\perp=I_D-\Pi}.
\tag{3.38}
$$

下面直接使用附录 C 证明的四个结论：

$$
\begin{aligned}
&\Pi^\top=\Pi,\quad \Pi^2=\Pi,\quad
\Pi_\perp^\top=\Pi_\perp,\quad \Pi_\perp^2=\Pi_\perp,\\[4pt]
&\Pi\Pi_\perp=0,\qquad
\|a\|^2=\|\Pi a\|^2+\|\Pi_\perp a\|^2,\\[4pt]
&\mathbb E[X^\top AX]=\operatorname{tr}(A\Sigma)
\quad\text{当 }\mathbb E[X]=0,\ \operatorname{Cov}(X)=\Sigma,\\[4pt]
&\operatorname{tr}(\Pi)=d,qquad
\operatorname{tr}(\Pi_\perp)=D-d.
\end{aligned}
\tag{3.39}
$$

这些结论不要求子空间与原始坐标轴对齐。对于任意朝向的 $d$ 维子空间，$\Pi$ 都是在原坐标系中完成相同的几何分解。

### 3.3.3 真实数据完全位于目标子空间

由 $x_1=Pz_1$ 和 $P^\top P=I_d$：

$$
\begin{aligned}
\Pi x_1
&=PP^\top Pz_1\\
&=P(P^\top P)z_1\\
&=Pz_1=x_1.
\end{aligned}
$$

因此：

$$
\boxed{\Pi x_1=x_1},qquad
\boxed{\Pi_\perp x_1=0}.
\tag{3.40}
$$

即使 $x_1$ 在原始坐标系中的 $D$ 个坐标全部非零，它在数据子空间之外仍然没有任何分量。坐标中非零元素的数量不等于子空间维度。

### 3.3.4 高斯噪声在子空间内外的能量

设：

$$
x_0\sim\mathcal N(0,I_D).
$$

对任意正交投影矩阵 $A$，由式（3.39）的二次型期望公式以及 $A^\top A=A$：

$$
\mathbb E\|Ax_0\|^2
=\mathbb E[x_0^\top Ax_0]
=\operatorname{tr}(A).
\tag{3.41}
$$

分别令 $A=\Pi$ 和 $A=\Pi_\perp$，得到：

$$
\boxed{
\mathbb E\|\Pi x_0\|^2=d,qquad
\mathbb E\|\Pi_\perp x_0\|^2=D-d.
}
\tag{3.42}
$$

这说明标准高斯在每个正交方向上都具有单位方差，所以它在一个 $k$ 维子空间中的平均平方投影长度正好是 $k$。结论只取决于维数，不取决于子空间朝向。

这里的“能量”指平方欧氏范数。式（3.42）是期望关系，并不表示每个噪声样本的平方范数都恰好等于相应维数。

### 3.3.5 条件速度在数据子空间之外的能量

把条件速度 $v=x_1-x_0$ 投影到两个正交子空间：

$$
\begin{aligned}
\Pi v
&=\Pi(x_1-x_0)
=x_1-\Pi x_0,\\[4pt]
\Pi_\perp v
&=\Pi_\perp(x_1-x_0)
=-\Pi_\perp x_0.
\end{aligned}
\tag{3.43}
$$

于是：

$$
\boxed{
v=
\underbrace{(x_1-\Pi x_0)}_{\text{数据子空间内}}
-
\underbrace{\Pi_\perp x_0}_{\text{数据子空间外}}
}.
\tag{3.44}
$$

两部分彼此正交，因此：

$$
\|v\|^2
=\|x_1-\Pi x_0\|^2
+\|\Pi_\perp x_0\|^2.
\tag{3.45}
$$

更关键的是，式（3.43）给出了逐样本恒等式：

$$
\|\Pi_\perp v\|^2=\|\Pi_\perp x_0\|^2.
$$

取期望即可得到核心结论：

$$
\boxed{
\mathbb E\|\Pi_\perp v\|^2=D-d.
}
\tag{3.46}
$$

这是根据论文 Toy Experiment 的构造补充得到的数学分析，并不是论文逐式给出的定理。

### 3.3.6 条件速度的完整能量分解

若 $x_0$ 与 $x_1$ 独立，且 $\mathbb E[x_0]=0$，则交叉项期望为零：

$$
\mathbb E[x_1^\top x_0]=0.
$$

因此：

$$
\boxed{
\mathbb E\|v\|^2
=\mathbb E\|x_1\|^2+D.
}
\tag{3.47}
$$

结合式（3.46）的正交补能量：

$$
\boxed{
\mathbb E\|\Pi v\|^2
=\mathbb E\|x_1\|^2+d.
}
\tag{3.48}
$$

最终得到：

$$
\boxed{
\underbrace{\mathbb E\|v\|^2}_{\text{速度总能量}}
=
\underbrace{\big(\mathbb E\|x_1\|^2+d\big)}_{\text{数据子空间内}}
+
\underbrace{(D-d)}_{\text{数据子空间外}}.
}
\tag{3.49}
$$

当 $d$ 固定而 $D$ 增大时，条件速度目标增长的能量主要来自数据子空间之外的噪声分量。这个能量结论解释了任务结构，但它本身不能证明某种神经网络一定无法预测速度。

### 3.3.7 为什么 x-prediction 可以减轻表示瓶颈？

Toy Experiment 使用隐藏宽度 $H=256$ 的网络。当 $D=512$ 时：

$$
H<D.
$$

直接预测 $x_1$ 时，网络的学习目标始终位于二维子空间。完整条件速度可以在网络外通过解析关系恢复：

$$
\boxed{
\hat v=\frac{\hat x_1-x_t}{1-t}.
}
\tag{3.50}
$$

这个表达式直接保留了原始高维输入 $x_t$，所以网络的隐藏表示无需重新编码所有高维噪声分量。它可以主要学习数据的低维结构，再由解析残差项补回速度。

还可以把这一点直接投影到正交补空间。由 $\Pi_\perp x_1=0$：

$$
\Pi_\perp x_t=(1-t)\Pi_\perp x_0,
$$

所以：

$$
\boxed{
\Pi_\perp v
=-\Pi_\perp x_0
=-\frac{\Pi_\perp x_t}{1-t}.
}
\tag{3.51}
$$

这表明速度的高维正交补分量并非统计上不可预测，而是可以从输入解析恢复。论文观察到的差异更准确地指向**有限容量网络的参数化与内部表示负担**。

### 3.3.8 实验设置、结果与能量对照

论文实验设置为：

- 内在维度：$d=2$。
- 环境维度：$D\in\{2,8,16,512\}$。
- 嵌入：随机生成并固定的列正交矩阵 $P$，模型并不知道 $P$。
- 网络：5 层 ReLU MLP，隐藏宽度为 256。
- 比较对象：x-prediction、$\epsilon$-prediction、v-prediction。
- 三种参数化统一使用 v-loss。

由式（3.42）和式（3.46），理论能量对照为：

| 环境维度 $D$ | 子空间维度 $d$ | 子空间内噪声平均平方能量 | 子空间外噪声与速度平均平方能量 |
|---:|---:|---:|---:|
| 2 | 2 | 2 | 0 |
| 8 | 2 | 2 | 6 |
| 16 | 2 | 2 | 14 |
| 512 | 2 | 2 | 510 |

论文观察到：

| 环境维度 | 实验现象 |
|---|---|
| $D=2$ | 三种预测方式均能较好生成 |
| $D=8$ | 不同预测方式开始出现差异 |
| $D=16$ | $\epsilon$-pred、v-pred 明显变难 |
| $D=512$ | $\epsilon$-pred、v-pred 严重失败，x-pred 仍能正常生成 |

因此，Toy Experiment 支持如下解释：当真实数据维数保持为 2 而环境维数不断升高时，直接预测干净数据允许有限容量网络集中表示低维数据结构；直接预测噪声或速度则要求网络自身处理越来越多的数据子空间外分量。

需要保留两个边界：

1. 这是特定网络、数据构造和训练设置下的实验结论，并非所有架构上的普遍定理。
2. $D-d$ 的推导解释了目标的几何和能量结构，不能单独推出某种参数化必然训练失败。

### 3.3.9 更一般的噪声协方差

若噪声只要求零均值，协方差为 $\Sigma$：

$$
\mathbb E[x_0]=0,qquad \operatorname{Cov}(x_0)=\Sigma,
$$

则同样由二次型期望公式得到：

$$
\boxed{
\mathbb E\|\Pi_\perp v\|^2
=\operatorname{tr}\big((I_D-\Pi)\Sigma\big).
}
\tag{3.52}
$$

对于各向同性噪声 $\Sigma=\sigma^2I_D$：

$$
\boxed{
\mathbb E\|\Pi_\perp v\|^2=\sigma^2(D-d).
}
\tag{3.53}
$$

如果 $\Sigma$ 不是各向同性的，结果通常还会依赖目标子空间的朝向。由此可见，使 $D-d$ 结论与子空间朝向无关的关键是单位协方差或更一般的各向同性，而不只是“噪声服从高斯分布”。

---

# 4. “Just Image Transformers” for Diffusion

第三节通过 Toy Experiment 验证了作者的核心假设。

第四节则进一步回答：

> 如果直接预测干净数据真的能够缓解高维表示瓶颈，那么能否使用最普通的 Vision Transformer，直接在原始像素空间训练扩散生成模型？

作者的答案是肯定的，并由此提出 JiT。

## 4.1 Just Image Transformers：模型架构

### 4.1.1 整体结构

JiT 的架构非常简单，可以理解为：

$$
\boxed{
\text{Raw Image}
\rightarrow
\text{Patchify}
\rightarrow
\text{Linear Embedding}
\rightarrow
\text{Transformer}
\rightarrow
\text{Linear Head}
\rightarrow
\hat x_1
}
\tag{4.1}
$$

其中，输入实际上是加噪图像 $x_t$，而不是干净图像。

与常见的 Latent Diffusion 不同，JiT 不需要：

$$
\text{Image}\rightarrow\text{VAE Encoder}\rightarrow\text{Latent}
$$

也不需要在生成结束后使用 VAE Decoder。

### 4.1.2 Patch Embedding

假设输入图像：

$$
x_t\in\mathbb R^{B\times H\times W\times3}
$$

其中：

- $B$：Batch Size。
- $H,W$：图像高度和宽度。
- 3：RGB 通道。

将图像划分为：

$$
p\times p
$$

的非重叠 patch。

那么 token 数量：

$$
\boxed{
N=\frac Hp\frac Wp
}
\tag{4.2}
$$

每个 patch 的原始维度：

$$
\boxed{
D_{\mathrm{patch}}=p^2\times3
}
\tag{4.3}
$$

例如，ImageNet 256×256：

$$
p=16
$$

则：

$$
N=16\times16=256
$$

$$
D_{\mathrm{patch}}=16\times16\times3=768
$$

因此：

$$
x_{\mathrm{patch}}\in\mathbb R^{B\times256\times768}
$$

接着通过线性映射：

$$
h=x_{\mathrm{patch}}W_{\mathrm{embed}}+b
$$

其中：

$$
W_{\mathrm{embed}}\in\mathbb R^{768\times H_{\mathrm{hidden}}}
$$

对于 JiT-B：

$$
H_{\mathrm{hidden}}=768
$$

因此：

$$
h\in\mathbb R^{B\times256\times768}
$$

然后加入位置编码，输入 Transformer。

### 4.1.3 Transformer 主干

论文的基础模型采用普通 ViT 风格的 Transformer。

在基础消融实验中，使用 adaLN-Zero 注入时间和类别条件。

每个 Transformer Block 主要包含：

$$
\text{Self-Attention}
$$

和：

$$
\text{FFN}
$$

最终通过线性层将每个 token 映射回像素 patch：

$$
\hat x_{\mathrm{patch}}
=
h_{\mathrm{out}}W_{\mathrm{out}}+b_{\mathrm{out}}
$$

其中：

$$
W_{\mathrm{out}}
\in
\mathbb R^{H_{\mathrm{hidden}}\times D_{\mathrm{patch}}}
$$

最后执行 Unpatchify，恢复完整图像：

$$
\boxed{
\hat x_1\in\mathbb R^{B\times H\times W\times3}
}
\tag{4.4}
$$

注意：

**网络最后一层直接输出的是干净图像，而不是速度或者噪声。**

这才是 JiT 与常规 velocity-prediction DiT 的核心区别。

---

## 4.2 What to Predict by the Network?

这一节通过大量消融实验研究：

**神经网络到底应该预测什么？**

作者并不是只比较三种预测方式，而是进一步排除其他可能的解释。

### 4.2.1 实验一：九种 Prediction / Loss 组合

论文比较两个设置。

**高维设置：**

ImageNet 256×256，JiT-B/16。

每个 patch：

$$
D_{\mathrm{patch}}=768
$$

Transformer 隐藏维度：

$$
H_{\mathrm{hidden}}=768
$$

**低维设置：**

ImageNet 64×64，JiT-B/4。

每个 patch：

$$
D_{\mathrm{patch}}=48
$$

隐藏维度仍然：

$$
H_{\mathrm{hidden}}=768
$$

两个设置的 token 数量均为256，因此可以在保持序列长度一致的情况下比较不同 patch 维度的影响。

#### 高维实验结果：Table 2(a)

| Loss Space | x-pred | ε-pred | v-pred |
|---|---:|---:|---:|
| x-loss | 10.14 | 379.21 | 107.55 |
| ε-loss | 10.45 | 394.58 | 126.88 |
| v-loss | **8.62** | 372.38 | 96.53 |

FID 越低越好。

这个实验有两个特别重要的观察。

**第一个观察：固定 Loss Space，改变 Prediction Space，结果差异巨大。**

例如固定 v-loss：

$$
L_v=\mathbb E\|\hat v-v\|^2
$$

结果：

$$
\mathrm{FID}_{x\text{-pred}}=8.62
$$

$$
\mathrm{FID}_{\epsilon\text{-pred}}=372.38
$$

$$
\mathrm{FID}_{v\text{-pred}}=96.53
$$

这意味着：

$$
\boxed{
\text{相同的 Loss Space，并不能保证不同 Prediction Space 的性能接近}
}
\tag{4.5}
$$

**第二个观察：固定 x-prediction，改变 Loss Space，模型都能正常工作。**

结果分别为：

$$
10.14,\quad10.45,\quad8.62
$$

说明：

$$
\boxed{
\text{x-prediction 的优势并不完全来自 Loss Weighting}
}
\tag{4.6}
$$

#### 低维实验结果：Table 2(b)

| Loss Space | x-pred | ε-pred | v-pred |
|---|---:|---:|---:|
| x-loss | 5.76 | 6.20 | 6.12 |
| ε-loss | 3.56 | 4.02 | 3.76 |
| v-loss | 3.55 | 3.63 | **3.46** |

可以发现，三种 Prediction Space 的性能差异明显缩小。

例如固定 v-loss：

$$
3.55,\quad3.63,\quad3.46
$$

甚至 v-prediction 略好于 x-prediction。

因此，论文的结论不是：

$$
x\text{-prediction 永远最好}
$$

而是：

$$
\boxed{
\text{当观测维度较高、网络存在表示瓶颈时，x-prediction 的优势更加明显}
}
\tag{4.7}
$$

这一点与第三节 Toy Experiment 一致。[arXiv](https://arxiv.org/html/2511.13720?utm_source=chatgpt.com)

---

### 4.2.2 实验二：Loss Weighting 能否解决问题？

作者进一步提出：

既然不同 Loss Space 对应不同的时间权重，那么能否通过调整损失权重，让 ε-prediction 或 v-prediction 也正常训练？

实际上，前面的九种组合已经提供了答案。

例如固定 v-prediction：

$$
\hat v=f_\theta(x_t,t)
$$

分别使用：

$$
L_x,\quad L_\epsilon,\quad L_v
$$

结果为：

$$
107.55,\quad126.88,\quad96.53
$$

虽然不同 Loss Space 会影响性能，但都无法接近：

$$
x\text{-pred}+v\text{-loss}=8.62
$$

因此作者认为：

**高维预测失败的主要原因不能仅仅归结为时间步损失权重不合理。**

这也说明：

$$
\boxed{
\text{Prediction Space 与 Loss Space 是两个不同的设计问题}
}
\tag{4.8}
$$

---

### 4.2.3 实验三：提高噪声水平能否解决问题？

一些像素空间扩散模型发现，提高训练噪声水平有助于高分辨率生成。

因此作者进一步研究：

如果让模型训练时看到更多高噪声样本，是否可以解决 ε-prediction 和 v-prediction 的问题？

论文使用 Logit-Normal 分布采样时间：

$$
s\sim\mathcal N(\mu,\sigma^2)
$$

$$
\boxed{
t=\operatorname{sigmoid}(s)
=
\frac{1}{1+e^{-s}}
}
\tag{4.9}
$$

由于：

$$
x_t=tx_1+(1-t)x_0
$$

因此：

- $t$ 越小，噪声占比越高。
- $t$ 越大，真实数据占比越高。

降低 $\mu$，会使时间分布向较小的 $t$ 移动。

也就是增加训练中的噪声水平。

论文 Table 3 的结果：

| $\mu$ | x-pred | ε-pred | v-pred |
|---|---:|---:|---:|
| 0.0 | 14.44 | 464.25 | 120.03 |
| -0.4 | 9.79 | 372.91 | 109.93 |
| -0.8 | **8.62** | 372.36 | **96.53** |
| -1.2 | 8.99 | 355.25 | 106.85 |

可以发现：

适当提高噪声水平，确实改善了 x-prediction。

但即使改变时间采样分布，ε-prediction 和 v-prediction 仍然严重退化。

因此：

$$
\boxed{
\text{调整 Noise Schedule 也不能充分解决高维预测瓶颈}
}
\tag{4.10}
$$

这里需要区分两个概念：

- Loss Weighting：给不同时间步的误差赋予不同权重。
- Time Sampling：改变训练时不同时间步被采样的概率。

二者都会影响训练中不同时间区间的有效监督强度，但并不完全相同。

---

### 4.2.4 实验四：增加隐藏维度是否必要？

这是论文一个很有意思的结论。

假设模型输入 patch：

$$
D_{\mathrm{patch}}=3072
$$

但是 Transformer hidden dimension：

$$
H_{\mathrm{hidden}}=768
$$

那么：

$$
H_{\mathrm{hidden}}<D_{\mathrm{patch}}
$$

直觉上，模型可能存在严重的信息瓶颈。

传统思路是增加隐藏维度，例如：

$$
768\rightarrow3072
$$

但是作者认为：

如果模型直接预测干净数据，未必需要这样做。

因为真实数据具有低维结构，网络不一定需要保存完整的3072维信息。

后续第五节的高分辨率实验表明：

即使：

$$
D_{\mathrm{patch}}=12288
$$

而：

$$
H_{\mathrm{hidden}}=768
$$

JiT 仍然能够生成具有一定质量的图像。

因此：

$$
\boxed{
H_{\mathrm{hidden}}\geq D_{\mathrm{patch}}
}
$$

并不是 x-prediction 正常工作的必要条件。

---

### 4.2.5 实验五：Bottleneck 甚至可能有利

作者进一步提出一个看起来有些反直觉的问题：

> 如果 x-prediction 只需要学习真实数据的低维结构，那么主动压缩输入表示，会不会反而改善性能？

为此，作者修改 Patch Embedding。

原本是：

$$
\mathbb R^{768}
\rightarrow
\mathbb R^{768}
$$

现在改为：

$$
\boxed{
\mathbb R^{768}
\rightarrow
\mathbb R^{d'}
\rightarrow
\mathbb R^{768}
}
\tag{4.11}
$$

其中：

$$
d'<768
$$

并且中间没有加入非线性激活。

因此，这两个线性层等价于一个低秩线性映射。

设：

$$
W_1\in\mathbb R^{768\times d'}
$$

$$
W_2\in\mathbb R^{d'\times768}
$$

那么：

$$
h=xW_1W_2
$$

令：

$$
W=W_1W_2
$$

由矩阵秩的性质：

$$
\operatorname{rank}(AB)
\leq
\min(\operatorname{rank}(A),\operatorname{rank}(B))
$$

得到：

$$
\boxed{
\operatorname{rank}(W)\leq d'
}
\tag{4.12}
$$

因此，输入映射的有效秩被限制在 $d'$ 以内。

作者发现：

- 即使 Bottleneck Dimension 降低到16，模型仍能正常工作。
- 在较宽的维度范围内，引入 Bottleneck 可以改善 FID。
- 部分设置的 FID 改善幅度达到约1.3。

这个实验支持了作者的观点：

**神经网络可以通过低维内部表示学习高维数据中的有效结构，额外保留所有高维输入信息并不一定有利。**

但要注意，Bottleneck 改善性能也可能涉及正则化、优化和模型归纳偏置，不能仅凭实验就断言其维度等于数据的真实内在维度。[arXiv](https://arxiv.org/html/2511.13720?utm_source=chatgpt.com)

---

## 4.3 Our Algorithm：完整训练与推理算法

这一节给出 JiT 的最终算法。

### 4.3.1 训练过程

第一步，从真实数据分布采样：

$$
x_1\sim p_{\mathrm{data}}
$$

第二步，采样高斯噪声：

$$
x_0\sim\mathcal N(0,I)
$$

第三步，采样时间：

$$
t\sim p(t)
$$

第四步，构造中间状态：

$$
\boxed{
x_t=tx_1+(1-t)x_0
}
\tag{4.13}
$$

第五步，计算真实速度：

$$
\boxed{
u=x_1-x_0
}
\tag{4.14}
$$

第六步，网络直接预测真实数据：

$$
\boxed{
\hat x_1=f_\theta(x_t,t)
}
\tag{4.15}
$$

第七步，将预测数据转换为速度：

$$
\boxed{
\hat u=\frac{\hat x_1-x_t}{1-t}
}
\tag{4.16}
$$

第八步，计算速度损失：

$$
\boxed{
L=
\mathbb E\|\hat u-u\|^2
}
\tag{4.17}
$$

这就是论文的核心训练算法。

注意：

虽然网络直接预测的是 $x_1$，但并没有直接使用：

$$
\|\hat x_1-x_1\|^2
$$

而是使用：

$$
\|\hat u-u\|^2
$$

所以最终配置是：

$$
\boxed{
\text{x-prediction}+\text{v-loss}
}
\tag{4.18}
$$

### 4.3.2 为什么可以使用速度损失？

根据前面的推导：

$$
\hat u-u
=
\frac{\hat x_1-x_1}{1-t}
$$

因此：

$$
L
=
\mathbb E
\left[
\frac{1}{(1-t)^2}
\|\hat x_1-x_1\|^2
\right]
\tag{4.19}
$$

所以本质上仍然在监督真实数据预测，只是不同时间步的误差权重不同。

这里还可以进一步计算网络输出的梯度。

对于单个样本，定义：

$$
L=
\frac{1}{(1-t)^2}
\|\hat x_1-x_1\|^2
$$

那么：

$$
\boxed{
\frac{\partial L}{\partial\hat x_1}
=
\frac{2}{(1-t)^2}(\hat x_1-x_1)
}
\tag{4.20}
$$

因此，靠近 $t=1$ 时，相同大小的 $x_1$ 预测误差会产生更大的梯度。

这解释了为什么需要注意端点数值稳定性。

### 4.3.3 分母截断

当：

$$
t\rightarrow1
$$

有：

$$
1-t\rightarrow0
$$

于是：

$$
\hat u=\frac{\hat x_1-x_t}{1-t}
$$

可能产生较大的数值。

论文采用：

$$
\boxed{
d_t=\max(1-t,0.05)
}
\tag{4.21}
$$

在实现速度转换时使用：

$$
\hat u=\frac{\hat x_1-x_t}{d_t}
$$

这样避免除以接近零的数。

需要注意，截断后的训练目标与理想公式存在差异。

如果真实速度和预测速度都使用同一个截断分母，则对应的有效数据误差权重为：

$$
\boxed{
w(t)=\frac{1}{\max(1-t,0.05)^2}
}
\tag{4.22}
$$

如果真实速度仍采用 $x_1-x_0$，而只对预测速度的转换分母进行截断，则端点附近不再具有这个简单的等价形式。论文伪代码与实现说明需要结合具体代码理解这一细节。

### 4.3.4 推理过程

推理时不需要真实数据。

首先采样：

$$
x_0\sim\mathcal N(0,I)
$$

然后在每个时间步：

$$
\hat x_1=f_\theta(x_t,t)
$$

计算：

$$
\hat u=\frac{\hat x_1-x_t}{1-t}
$$

最后进行数值积分：

$$
x_{t+\Delta t}=x_t+\Delta t\hat u
$$

重复直到：

$$
t=1
$$

即可得到生成样本。

**因此，JiT 并没有改变 Flow Matching 的 ODE 生成框架，只改变了神经网络的直接预测目标。**

---

## 4.4 “Just Advanced” Transformers

这一节研究：

如果 JiT 使用普通 Transformer，那么能否直接利用大语言模型领域成熟的架构改进？

作者引入：

- SwiGLU。
- RMSNorm。
- RoPE。
- QK-Norm。
- In-context Class Conditioning。

其中前四项都是通用 Transformer 组件。

### 4.4.1 SwiGLU

SwiGLU 可以写成：

$$
\operatorname{SwiGLU}(x)
=
\operatorname{SiLU}(xW_1)
\odot(xW_2)
$$

然后通过输出投影：

$$
y=\operatorname{SwiGLU}(x)W_3
$$

与普通 FFN 相比，它引入门控机制。

### 4.4.2 RMSNorm

RMSNorm 定义为：

$$
\boxed{
\operatorname{RMSNorm}(x)
=
\frac{x}
{\sqrt{\frac1d\sum_{i=1}^d x_i^2+\epsilon}}
\odot\gamma
}
\tag{4.23}
$$

其中：

- $d$：特征维度。
- $\gamma$：可学习缩放参数。
- $\epsilon$：数值稳定项。

与 LayerNorm 不同，RMSNorm 不进行均值中心化。

### 4.4.3 RoPE 与 QK-Norm

RoPE 将位置信息编码到 Query 和 Key 的旋转变换中。

QK-Norm 则对注意力中的 Query 和 Key 进行归一化，以改善注意力数值稳定性。

### 4.4.4 In-context Class Conditioning

传统 DiT 常通过 adaLN-Zero 注入类别条件。

作者进一步尝试将类别条件表示为多个 token，直接加入 Transformer 序列。

论文默认使用32个类别条件 token。

它们不是一开始就必须加入，而是可以在较深的 Transformer Block 中引入。

这种方式允许图像 token 与类别 token 通过注意力机制交互。

### 4.4.5 实验结果

论文 Table 4：

| 模型配置 | JiT-B/16 FID |
|---|---:|
| SwiGLU + RMSNorm | 7.48 |
| + RoPE + QK-Norm | 6.69 |
| + In-context Class Tokens | 5.49 |

如果进一步使用 CFG Interval，对应结果为：

$$
6.32\rightarrow5.44\rightarrow4.37
$$

说明：

**在解决 Prediction Space 问题之后，普通 Transformer 的通用架构改进也可以继续提升生成性能。**

这支持了作者希望构建通用 Diffusion + Transformer 框架的研究目标。[arxiv.org](https://arxiv.org/html/2511.13720?utm_source=chatgpt.com)

---

# 5. Comparisons：实验性能与扩展性

这一节主要研究三个问题：

1. JiT 能否直接生成高分辨率图像？
2. 模型参数量增加时，生成性能是否继续提升？
3. 与已有 Latent Diffusion 和 Pixel Diffusion 相比，JiT 的性能如何？

## 5.1 高分辨率生成

作者设计了三个模型：

| 分辨率 | 模型 | Patch 维度 | Hidden Dimension | FID |
|---|---|---:|---:|---:|
| 256×256 | JiT-B/16 | 768 | 768 | 4.37 |
| 512×512 | JiT-B/32 | 3072 | 768 | 4.64 |
| 1024×1024 | JiT-B/64 | 12288 | 768 | 4.82 |

这是论文 Table 5 的结果。

最重要的是：

随着分辨率增加，作者同步增加 patch size。

因此：

$$
N=\frac Hp\frac Wp
$$

始终保持：

$$
N=256
$$

例如：

$$
\frac{256}{16}=16
$$

$$
\frac{512}{32}=16
$$

$$
\frac{1024}{64}=16
$$

所以 Transformer 的序列长度始终为：

$$
16\times16=256
$$

### 为什么计算量没有随分辨率大幅增加？

标准 Self-Attention 的主要计算复杂度近似为：

$$
O(N^2H_{\mathrm{hidden}})
$$

其中：

- $N$：token 数量。
- $H_{\mathrm{hidden}}$：隐藏维度。

如果保持：

$$
N=256
$$

以及：

$$
H_{\mathrm{hidden}}=768
$$

那么 Transformer 主干的计算量基本不变。

增加分辨率主要影响：

- 输入 Patch Embedding。
- 输出 Linear Head。

因此论文报告的计算量仅从约25 GFLOPs 增加到30 GFLOPs。

这不是说高分辨率生成完全没有额外成本，而是说：

**通过增加 patch size，可以避免 Transformer 主干计算量随着图像像素数量急剧增加。**

不过，这样做的前提是模型能够处理越来越高维的 patch。

而作者认为，x-prediction 正是使这种设计可行的关键。

---

## 5.2 模型扩展性

论文进一步比较四种模型：

- JiT-B。
- JiT-L。
- JiT-H。
- JiT-G。

在 ImageNet 256×256 上：

| 模型 | 200 Epoch FID | 600 Epoch FID |
|---|---:|---:|
| JiT-B/16 | 4.37 | 3.66 |
| JiT-L/16 | 2.79 | 2.36 |
| JiT-H/16 | 2.29 | 1.86 |
| JiT-G/16 | 2.15 | **1.82** |

在 ImageNet 512×512 上：

| 模型 | 200 Epoch FID | 600 Epoch FID |
|---|---:|---:|
| JiT-B/32 | 4.64 | 4.02 |
| JiT-L/32 | 3.06 | 2.53 |
| JiT-H/32 | 2.51 | 1.94 |
| JiT-G/32 | 2.11 | **1.78** |

可以发现：

随着模型规模增加，FID 整体持续改善。

并且，即使在512×512分辨率下，patch 维度达到3072，模型隐藏维度仍然低于 patch 维度。

这说明：

$$
\boxed{
\text{高维 Patch}+\text{低维隐藏表示}
}
$$

并不会必然导致生成失败。

作者认为，这与低维流形假设相符。

---

## 5.3 与已有方法的比较

论文 Table 7 和 Table 8 将 JiT 与多种生成模型进行比较。

例如 ImageNet 256×256：

| 模型 | 生成空间 | FID |
|---|---|---:|
| DiT-XL/2 | VAE Latent | 2.27 |
| SiT-XL/2 | VAE Latent | 2.06 |
| REPA | VAE Latent | 1.42 |
| JiT-H/16 | Raw Pixels | 1.86 |
| JiT-G/16 | Raw Pixels | 1.82 |

需要注意，这些模型的参数量、训练资源、预训练依赖和辅助损失并不完全一致，因此不能把 FID 直接视为严格控制变量的比较。

作者真正想强调的是：

**JiT 在不依赖 VAE、感知损失和自监督预训练的情况下，仍然能够获得具有竞争力的图像生成结果。**

这说明，原始像素空间并不一定需要复杂的额外模块才能实现有效生成。

不过，论文并没有证明 JiT 在所有指标上全面超过已有方法。

例如，一些使用预训练表示的 Latent Diffusion 方法仍然具有更低的 FID。

因此更准确的结论是：

$$
\boxed{
\text{JiT 验证了直接在高维原始数据空间进行生成的可行性}
}
$$

而不是证明 Latent Diffusion 已经没有价值。[arXiv](https://arxiv.org/html/2511.13720?utm_source=chatgpt.com)

---

# 6. Discussion and Conclusion

论文最后重新回到最初的问题：

**为什么神经网络应该直接预测干净数据？**

作者认为，过去扩散模型研究更多关注概率模型本身，例如：

- Score Matching。
- Noise Schedule。
- SDE / ODE。
- 采样求解器。
- Loss Weighting。

但是对神经网络自身的表示能力关注相对不足。

从概率建模角度：

$$
x,\epsilon,v
$$

可以通过线性变换互相恢复。

但从有限容量神经网络的角度，这些预测任务可能具有完全不同的表示难度。

真实数据具有结构性，而噪声是高维随机变量。

因此：

$$
\boxed{
\text{神经网络有限的表示能力，应尽可能用于学习真实数据结构}
}
\tag{6.1}
$$

而不是要求网络直接重建所有高维噪声信息。

作者进一步认为，这种思想可能适用于其他高维自然数据，例如：

- 分子。
- 蛋白质。
- 气象数据。
- 其他难以设计专用 tokenizer 的科学数据。

但这些属于论文提出的潜在研究方向，并不是已经全面验证的结论。

---

# 7. Appendix A：实现细节

附录 A 提供了复现实验所需要的主要超参数。

## 7.1 时间采样分布

论文采用：

$$
s\sim\mathcal N(\mu,\sigma^2)
$$

$$
t=\operatorname{sigmoid}(s)
$$

默认：

$$
\boxed{
\mu=-0.8,\qquad\sigma=0.8
}
\tag{7.1}
$$

这里可以推导出 $t$ 的概率密度。

首先：

$$
s=\log\frac{t}{1-t}
$$

因此：

$$
\frac{ds}{dt}
=
\frac1t+\frac1{1-t}
$$

整理：

$$
\boxed{
\frac{ds}{dt}=\frac1{t(1-t)}
}
\tag{7.2}
$$

由随机变量变换公式：

$$
p_t(t)=p_s(s(t))
\left|\frac{ds}{dt}\right|
$$

而：

$$
p_s(s)
=
\frac1{\sigma\sqrt{2\pi}}
\exp\left(
-\frac{(s-\mu)^2}{2\sigma^2}
\right)
$$

所以：

$$
\boxed{
p_t(t)
=
\frac1{\sigma\sqrt{2\pi}\,t(1-t)}
\exp\left[
-\frac{
\left(\log\frac{t}{1-t}-\mu\right)^2
}{2\sigma^2}
\right]
}
\tag{7.3}
$$

其中：

$$
0<t<1
$$

这就是 Logit-Normal 分布。

因此，训练时的完整期望实际上是：

$$
L=
\int_0^1
p_t(t)
\mathbb E_{x_0,x_1}
\left[
w(t)\|\hat x_1-x_1\|^2
\right]dt
\tag{7.4}
$$

这说明：

**Loss Space 决定的 $w(t)$ 与时间采样分布 $p_t(t)$ 会共同决定不同时间步的有效训练权重。**

---

## 7.2 高分辨率噪声缩放

论文在不同分辨率下采用不同噪声尺度。

对于256×256：

$$
x_0\sim\mathcal N(0,I)
$$

对于512×512：

$$
x_0\sim\mathcal N(0,4I)
$$

对于1024×1024：

$$
x_0\sim\mathcal N(0,16I)
$$

也就是噪声标准差分别为：

$$
1,\quad2,\quad4
$$

作者这样设计是为了近似维持不同分辨率下的信噪比关系。

这是论文中的实验设计，不代表所有高分辨率 Flow Matching 都必须按分辨率线性放大噪声标准差。

---

## 7.3 主要训练超参数

| 参数 | 设置 |
|---|---|
| Optimizer | Adam |
| Batch Size | 1024 |
| Learning Rate | $2\times10^{-4}$ |
| Weight Decay | 0 |
| Warmup | 5 Epoch |
| Training Epoch | 200 / 600 |
| Time Distribution | Logit-Normal |
| $\mu,\sigma$ | -0.8，0.8 |
| EMA Decay | 0.9996 / 0.9998 / 0.9999 |
| ODE Solver | Heun |
| ODE Steps | 50 |
| CFG Scale | 1.0–4.0 |
| CFG Drop Probability | 0.1 |

其中，论文对不同实验配置搜索合适的 EMA 和 CFG Scale，以减少超参数选择对消融结论的干扰。[arXiv](https://arxiv.org/html/2511.13720?utm_source=chatgpt.com)

---

# 8. Appendix B：额外实验

这一部分非常重要，因为它进一步检验了论文核心结论是否成立。

## 8.1 Training Loss and Denoised Images

作者比较：

$$
x\text{-prediction}+v\text{-loss}
$$

与：

$$
v\text{-prediction}+v\text{-loss}
$$

注意，两种方法的 Loss Space 完全相同。

因此，训练损失可以直接比较。

论文 Figure 7 显示：

**直接预测速度的模型，其训练损失比直接预测干净数据的模型高约25%。**

此外，ε-prediction 的训练损失更高，并存在不稳定现象。

这说明：

高维生成失败不只是采样求解器的问题。

在训练阶段，模型就已经难以准确学习高维预测目标。

作者还比较了不同噪声水平下的单步去噪图像。

x-prediction 可以恢复较合理的图像结构，而 v-prediction 对应的去噪结果存在明显伪影。

这些单步预测误差在多步 ODE 采样中还可能进一步传播。

因此：

$$
\boxed{
\text{高维预测目标的训练困难，会进一步影响最终生成质量}
}
\tag{8.1}
$$

---

## 8.2 Pre-conditioner：为什么 EDM 式预条件化也不够？

这一节值得详细分析。

因为它涉及一个容易混淆的问题：

> EDM 最终也预测干净数据，为什么作者仍然认为它与直接 x-prediction 不同？

### 8.2.1 EDM 式预条件化

一般可以写为：

$$
\boxed{
\hat x_1
=
c_{\mathrm{skip}}(t)x_t
+
c_{\mathrm{out}}(t)f_\theta(x_t,t)
}
\tag{8.2}
$$

其中：

- $c_{\mathrm{skip}}(t)$：输入残差系数。
- $c_{\mathrm{out}}(t)$：网络输出系数。
- $f_\theta$：神经网络直接输出。

如果：

$$
c_{\mathrm{skip}}=0
$$

$$
c_{\mathrm{out}}=1
$$

那么：

$$
\hat x_1=f_\theta(x_t,t)
$$

这才是论文所定义的直接 x-prediction。

但是当：

$$
c_{\mathrm{skip}}\neq0
$$

时，网络直接输出的不是干净数据。

### 8.2.2 推导网络真正需要预测什么

从式（8.2）：

$$
\hat x_1
=
c_{\mathrm{skip}}x_t
+
c_{\mathrm{out}}f_\theta
$$

移项：

$$
c_{\mathrm{out}}f_\theta
=
\hat x_1-c_{\mathrm{skip}}x_t
$$

所以：

$$
\boxed{
f_\theta
=
\frac{
\hat x_1-c_{\mathrm{skip}}x_t
}{
c_{\mathrm{out}}
}
}
\tag{8.3}
$$

如果理想情况下：

$$
\hat x_1=x_1
$$

并代入：

$$
x_t=tx_1+(1-t)x_0
$$

则：

$$
f_\theta^*
=
\frac{
x_1-c_{\mathrm{skip}}
[tx_1+(1-t)x_0]
}{
c_{\mathrm{out}}
}
$$

展开：

$$
f_\theta^*
=
\frac{
(1-tc_{\mathrm{skip}})x_1
-
(1-t)c_{\mathrm{skip}}x_0
}{
c_{\mathrm{out}}
}
$$

因此：

$$
\boxed{
f_\theta^*
=
\frac{1-tc_{\mathrm{skip}}}{c_{\mathrm{out}}}x_1
-
\frac{(1-t)c_{\mathrm{skip}}}{c_{\mathrm{out}}}x_0
}
\tag{8.4}
$$

这个公式说明：

只要：

$$
c_{\mathrm{skip}}\neq0
$$

并且：

$$
t\neq1
$$

网络直接输出的理想目标一般仍然包含噪声分量。

所以：

$$
\boxed{
\text{最终输出是去噪结果}
\neq
\text{神经网络直接预测干净数据}
}
\tag{8.5}
$$

这正是作者想强调的区别。

### 8.2.3 实验结果

论文比较：

- 直接 x-prediction。
- EDM-style Pre-conditioner。
- Linear Pre-conditioner。

在 ImageNet 256×256 上：

| Loss Space | 直接 x-pred | EDM-style | Linear |
|---|---:|---:|---:|
| x-loss | 10.14 | 28.94 | 39.50 |
| ε-loss | 10.45 | 72.05 | 67.56 |
| v-loss | **8.62** | 35.49 | 46.25 |

可以发现：

直接 x-prediction 明显更好。

但预条件化方法又比直接 ε-prediction 和 v-prediction 的严重失败结果有所改善。

作者推测，这可能是因为部分预条件化形式在高噪声区域更接近直接 x-prediction。

需要注意，EDM 原本使用的噪声过程与本文不同，因此这里是作者在本文设置下构造的 EDM-style 对照，并不是对原始 EDM 完整系统的直接性能比较。[arXiv](https://arxiv.org/html/2511.13720?utm_source=chatgpt.com)

---

## 8.3 Exploration: Classification Loss

作者进一步探索：

如果在 JiT 中增加分类损失，是否可以继续改善生成质量？

这部分不是论文主方法，而是额外实验。

核心思想是让模型隐藏特征承担一定的类别判别任务，从而提供额外监督。

可以抽象为：

$$
L_{\mathrm{total}}
=
L_v+\lambda L_{\mathrm{cls}}
\tag{8.6}
$$

其中：

$$
L_{\mathrm{cls}}
=
-\log p_\theta(y\mid h)
$$

作者通过额外分类头实现这一目标，并采取措施避免类别标签直接泄漏到分类预测中。

实验显示，额外分类监督具有改善生成性能的潜力。

但论文强调：

**主体实验并没有使用这种额外损失。**

因此，JiT 的主要结论并不依赖分类损失。

---

## 8.4 Cross-resolution Generation

作者还比较了跨分辨率生成。

例如：

- 在256分辨率训练模型，然后上采样到512。
- 在512分辨率训练模型，然后下采样到256。

论文结果：

| 方法 | FID |
|---|---:|
| JiT-G/16，256原生生成 | 1.82 |
| JiT-G/32，512生成后下采样到256 | 1.84 |
| JiT-G/16，256生成后上采样到512 | 2.45 |
| JiT-G/32，512原生生成 | 1.78 |

可以发现：

512模型生成后下采样，几乎可以达到256原生模型的效果。

但是256模型生成后上采样，效果明显不如512原生模型。

原因在于：

简单上采样无法恢复原始生成过程中不存在的高频细节。

这一实验也说明，JiT 的高分辨率生成并不只是对低分辨率图像进行简单插值。

---

## 8.5 Additional Metrics

论文最后补充 Precision、Recall 等指标。

这是因为 FID 虽然可以衡量生成分布与真实数据分布的差异，但不能完整描述生成质量与多样性。

例如：

- Precision：生成样本是否主要位于真实数据分布的高质量区域。
- Recall：生成模型是否覆盖了真实数据分布中的多样性。

这些指标用于补充 FID 的评价。

作者报告的结果说明，JiT 在生成质量和分布覆盖方面具有一定竞争力，但不同方法之间仍存在取舍。

---

# 9. 对整篇论文数学结论的进一步理解

这一部分不是原文新增章节，而是将前面的数学结果串联起来。

我认为理解这篇论文，最重要的是区分三个层次。

## 9.1 第一层：数学参数化等价

Flow Matching：

$$
x_t=tx_1+(1-t)x_0
$$

$$
u=x_1-x_0
$$

因此：

$$
\boxed{
u=\frac{x_1-x_t}{1-t}
}
\tag{9.1}
$$

这意味着：

只要知道：

$$
x_t,t
$$

网络预测：

$$
x_1
$$

或者预测：

$$
u
$$

在代数上都可以恢复相同的变量。

如果允许任意复杂的函数，那么可以通过变量转换建立对应关系。

因此，从理想函数空间角度，三种参数化可以表达相同的目标速度场。

---

## 9.2 第二层：有限容量网络的表示能力不同

假设模型具有隐藏表示：

$$
h_\theta(x_t,t)\in\mathbb R^H
$$

对于直接速度预测：

$$
\hat u=W_uh_\theta(x_t,t)
$$

网络需要通过隐藏表示构造完整速度。

对于直接数据预测：

$$
\hat x_1=W_xh_\theta(x_t,t)
$$

再通过：

$$
\hat u=\frac{\hat x_1-x_t}{1-t}
$$

恢复速度。

因此第二种方法实际上具有：

$$
\boxed{
\hat u
=
\frac{W_xh_\theta(x_t,t)}{1-t}
-
\frac{x_t}{1-t}
}
\tag{9.2}
$$

这一结构。

注意最后一项：

$$
-\frac{x_t}{1-t}
$$

不需要经过隐藏表示。

它直接利用原始输入。

因此：

**x-prediction 的优势不只是输出目标具有低维结构，还包括它能够通过解析残差路径保留高维输入信息。**

这两个因素共同解释了为什么有限容量网络可能更适合 x-prediction。

---

## 9.3 第三层：低维流形假设并不是严格的性能保证

假设：

$$
x_1\in\mathcal M
$$

其中：

$$
\dim(\mathcal M)=d\ll D
$$

这说明真实数据具有低维结构。

但是在 MSE 训练下，理想预测为：

$$
\boxed{
f^*(x_t,t)=\mathbb E[x_1\mid x_t,t]
}
\tag{9.3}
$$

而条件期望未必严格位于流形上。

例如真实数据只有两个可能值：

$$
x_1\in\{-1,+1\}
$$

如果某个条件下：

$$
P(x_1=-1\mid x_t)=P(x_1=+1\mid x_t)=0.5
$$

那么：

$$
\mathbb E[x_1\mid x_t]=0
$$

但：

$$
0\notin\{-1,+1\}
$$

因此，不能简单认为：

$$
\hat x_1\in\mathcal M
$$

必然成立。

更准确地说：

**x-prediction 的目标数据具有低维结构，网络可以利用这种结构，但普通 MSE 并没有显式保证预测结果严格位于数据流形上。**

同样，低维流形可能具有复杂的非线性几何结构，因此内在维度较低也不必然意味着学习简单。

这篇论文的贡献是通过理论直觉与实验说明，在所研究的高维生成架构中，直接预测干净数据确实能够显著缓解表示瓶颈。

---

# 10. 全文最终总结

这篇论文的逻辑可以整理成以下过程。

### 第一阶段：发现问题

传统扩散模型常预测：

$$
\epsilon
$$

或：

$$
v
$$

虽然它们与真实数据 $x$ 可以互相转换，但在高维空间中，直接预测高维噪声相关目标可能给有限容量网络带来较大压力。

### 第二阶段：提出解释

根据低维流形假设：

$$
\boxed{
d_{\mathrm{intrinsic}}\ll D_{\mathrm{ambient}}
}
$$

真实数据具有较低的有效自由度。

而：

$$
\epsilon\sim\mathcal N(0,I_D)
$$

是全维噪声。

因此：

$$
v=x-\epsilon
$$

也包含大量高维噪声信息。

### 第三阶段：提出方法

网络直接预测：

$$
\boxed{
\hat x_1=f_\theta(x_t,t)
}
$$

然后转换为速度：

$$
\boxed{
\hat u=\frac{\hat x_1-x_t}{1-t}
}
$$

训练采用：

$$
\boxed{
L=\mathbb E\|\hat u-u\|^2
}
$$

推理仍然使用：

$$
\boxed{
\frac{dx_t}{dt}=\hat u(x_t,t)
}
$$

因此最终方案是：

$$
\boxed{
\text{x-prediction}
+
\text{v-loss}
+
\text{Flow Matching ODE}
}
$$

### 第四阶段：实验验证

作者通过 Toy Experiment 和 ImageNet 实验说明：

- 低维情况下，三种预测方式通常都能工作。
- 高维情况下，直接预测噪声或速度可能严重退化。
- 仅改变 Loss Space 或训练噪声分布不能充分解决问题。
- 直接预测干净数据能够明显改善高维生成性能。
- 即使隐藏维度小于输入 patch 维度，模型仍然可以有效生成。
- 适当的 Bottleneck 甚至可能改善性能。

### 第五阶段：验证实际价值

作者构建 JiT，证明：

不使用 VAE、额外预训练和辅助损失，也可以使用普通 Transformer 直接在原始像素空间完成具有竞争力的图像生成。

---

## 我认为最值得记住的三个结论

**第一，Prediction Space 与 Loss Space 应当解耦。**

$$
\boxed{
\text{网络预测什么}
\neq
\text{在哪个空间计算 Loss}
}
$$

固定 Prediction Space 时，不同 Loss Space 主要对应不同的时间步误差加权。

**第二，数学等价不代表神经网络学习难度等价。**

$$
\boxed{
x\longleftrightarrow\epsilon\longleftrightarrow v
}
$$

虽然可以互相转换，但不同参数化对网络内部表示能力的要求可能不同。

**第三，低维流形结构可以帮助模型在高维空间中学习真实数据。**

$$
\boxed{
d_{\mathrm{intrinsic}}\ll D_{\mathrm{ambient}}
}
$$

当真实数据具有低维结构、模型又存在表示瓶颈时，直接预测干净数据可能比直接预测包含高维噪声的目标更有效。

**因此，这篇论文最核心的思想可以概括为：让神经网络将有限的表示能力用于学习真实数据的结构，而不是让它承担本可通过解析公式恢复的高维噪声信息。**

这也是论文标题 *Let Denoising Generative Models Denoise* 的真正含义。

**原文：** [Back to Basics: Let Denoising Generative Models Denoise（arXiv:2511.13720v2）](https://arxiv.org/html/2511.13720)

---

# 附录 C：Toy Experiment 所需的数学基础

本附录集中给出正文 3.3 节使用的线性代数与概率论基础。阅读正文时可以直接使用式（3.39）中的结论；这里负责说明这些结论为什么成立。

需要说明：下面关于投影矩阵、Gram 矩阵、坐标变换、协方差与能量分解的证明，是依据论文 Toy Experiment 的构造补充的数学分析，并非声称论文逐条给出了这些公式。


## C.1 环境空间与目标子空间

### C.1.1 环境空间

设真实数据表示为：

\[
x_1\in\mathbb R^D
\]

例如：

\[
D=512
\]

表示每个样本具有512个坐标。

但是，这不意味着样本具有512个独立自由度。

假设所有真实数据都位于一个 \(d\) 维线性子空间：

\[
\mathcal S\subseteq\mathbb R^D
\]

其中：

\[
d<D
\]

例如：

\[
d=2,\qquad D=512
\]

这意味着真实数据虽然使用512个坐标表示，但实际上可以通过两个独立参数确定。

这里要区分：

\[
\boxed{
\text{向量的坐标数量}\neq\text{向量所在子空间的维度}
}
\]

一个二维平面可以倾斜地放置在三维空间中。平面上的向量仍然有三个坐标，但只具有两个独立自由度。

同理，512维空间中也可以存在一个任意朝向的二维线性子空间。

### C.1.2 用基向量定义目标子空间

设：

\[
p_1,p_2,\ldots,p_d\in\mathbb R^D
\]

是目标子空间的一组线性无关基向量。

定义基矩阵：

\[
\boxed{
P=[p_1,p_2,\ldots,p_d]\in\mathbb R^{D\times d}
}
\tag{C.1.1}
\]

那么：

\[
\boxed{
\mathcal S=\operatorname{Col}(P)
}
\tag{C.1.2}
\]

其中 \(\operatorname{Col}(P)\) 表示矩阵 \(P\) 的列空间。

目标子空间中的任意向量都可以表示为：

\[
\boxed{
y=Pz,\qquad z\in\mathbb R^d
}
\tag{C.1.3}
\]

这里：

- \(z\)：低维坐标。
- \(P\)：将低维坐标嵌入高维环境空间。
- \(y\)：高维空间中位于目标子空间上的向量。

因此：

\[
P:\mathbb R^d\rightarrow\mathbb R^D
\]

**\(P\) 本身是基矩阵，不是投影矩阵。**

---

## C.2 正交投影为什么来自最小二乘？

这一部分先不假设基向量正交，从一般情况推导。

### C.2.1 定义投影问题

给定任意向量：

\[
a\in\mathbb R^D
\]

希望找到目标子空间：

\[
\mathcal S=\operatorname{Col}(P)
\]

中距离 \(a\) 最近的向量。

由于子空间中的任意向量都能写成：

\[
y=Pz
\]

因此求解：

\[
\boxed{
z^*=\arg\min_{z\in\mathbb R^d}\|a-Pz\|^2
}
\tag{C.2.1}
\]

这是标准的线性最小二乘问题。

其中：

\[
e=a-Pz
\]

是重建残差。

### C.2.2 推导正规方程

定义：

\[
L(z)=\|a-Pz\|^2
\]

展开：

\[
\begin{aligned}
L(z)
&=(a-Pz)^\top(a-Pz)\\
&=a^\top a-2z^\top P^\top a+z^\top P^\top Pz
\end{aligned}
\]

对 \(z\) 求梯度：

\[
\nabla_zL
=
-2P^\top a+2P^\top Pz
\]

令梯度为零：

\[
P^\top Pz=P^\top a
\]

得到正规方程：

\[
\boxed{
P^\top Pz^*=P^\top a
}
\tag{C.2.2}
\]

由于 \(P\) 列满秩，\(P^\top P\) 可逆：

\[
\boxed{
z^*=(P^\top P)^{-1}P^\top a
}
\tag{C.2.3}
\]

将最优坐标代回：

\[
y^*=Pz^*
\]

得到：

\[
\boxed{
y^*=P(P^\top P)^{-1}P^\top a
}
\tag{C.2.4}
\]

因此，定义正交投影矩阵：

\[
\boxed{
\Pi=P(P^\top P)^{-1}P^\top
}
\tag{C.2.5}
\]

就有：

\[
\boxed{
y^*=\Pi a
}
\tag{C.2.6}
\]

这证明了：

**正交投影可以看作一个最小二乘问题，其目标是寻找子空间中距离原向量最近的向量。**

### C.2.3 Gram 矩阵的作用

定义：

\[
\boxed{
G=P^\top P
}
\tag{2.7}
\]

它就是 Gram 矩阵。

其中：

\[
G_{ij}=p_i^\top p_j
\]

记录各基向量之间的内积。

当基向量不正交时，不能直接用 \(P^\top a\) 作为子空间坐标，而需要：

\[
z^*=G^{-1}P^\top a
\]

Gram 矩阵的逆负责修正基向量之间的非正交关系。

### C.2.4 标准正交基下的简化

Toy Experiment 使用满足以下条件的矩阵：

\[
\boxed{
P^\top P=I_d
}
\tag{2.8}
\]

也就是 \(P\) 的列向量构成一组标准正交基。

此时：

\[
G=I_d
\]

因此：

\[
z^*=P^\top a
\]

投影矩阵简化为：

\[
\boxed{
\Pi=PP^\top
}
\tag{2.9}
\]

这就是后面使用的投影矩阵。

注意：

\[
P^\top P=I_d
\]

但一般：

\[
PP^\top\neq I_D
\]

因为当 \(d<D\) 时，\(P\) 只张成整个高维空间的一部分。

---

## C.3 正交投影矩阵的性质

后续推导会反复使用三个性质。

### C.3.1 对称性

由于：

\[
\Pi=PP^\top
\]

所以：

\[
\Pi^\top=(PP^\top)^\top
\]

利用矩阵转置性质：

\[
(AB)^\top=B^\top A^\top
\]

得到：

\[
\Pi^\top=PP^\top
\]

因此：

\[
\boxed{
\Pi^\top=\Pi
}
\tag{C.3.1}
\]

### C.3.2 幂等性

计算：

\[
\Pi^2=PP^\top PP^\top
\]

整理：

\[
\Pi^2=P(P^\top P)P^\top
\]

由于：

\[
P^\top P=I_d
\]

所以：

\[
\Pi^2=PI_dP^\top
\]

最终：

\[
\boxed{
\Pi^2=\Pi
}
\tag{C.3.2}
\]

几何含义是：

**一个向量投影到目标子空间后，再投影一次不会改变结果。**

### C.3.3 正交补投影

定义：

\[
\boxed{
\Pi_\perp=I_D-\Pi
}
\tag{C.3.3}
\]

它是到目标子空间正交补：

\[
\mathcal S^\perp
\]

的正交投影矩阵。

验证幂等性：

\[
\begin{aligned}
\Pi_\perp^2
&=(I-\Pi)^2\\
&=I-2\Pi+\Pi^2\\
&=I-\Pi\\
&=\Pi_\perp
\end{aligned}
\]

所以：

\[
\boxed{
\Pi_\perp^2=\Pi_\perp
}
\tag{C.3.4}
\]

同时：

\[
\Pi\Pi_\perp
=
\Pi(I-\Pi)
=
\Pi-\Pi^2
=
0
\]

因此：

\[
\boxed{
\Pi\Pi_\perp=0
}
\tag{C.3.5}
\]

这说明两个投影对应的子空间相互正交。

---

## C.4 向量的正交分解与勾股定理

对于任意：

\[
a\in\mathbb R^D
\]

有：

\[
a=I_Da
\]

而：

\[
I_D=\Pi+(I_D-\Pi)
\]

所以：

\[
\boxed{
a=\Pi a+\Pi_\perp a
}
\tag{C.4.1}
\]

定义：

\[
a_\parallel=\Pi a
\]

\[
a_\perp=\Pi_\perp a
\]

于是：

\[
\boxed{
a=a_\parallel+a_\perp
}
\tag{C.4.2}
\]

其中：

\[
a_\parallel\in\mathcal S
\]

\[
a_\perp\in\mathcal S^\perp
\]

两者正交：

\[
\begin{aligned}
a_\parallel^\top a_\perp
&=(\Pi a)^\top(\Pi_\perp a)\\
&=a^\top\Pi^\top\Pi_\perp a\\
&=a^\top\Pi\Pi_\perp a\\
&=0
\end{aligned}
\]

因此：

\[
\boxed{
a_\parallel^\top a_\perp=0
}
\tag{C.4.3}
\]

由此得到高维空间中的勾股定理：

\[
\begin{aligned}
\|a\|^2
&=\|a_\parallel+a_\perp\|^2\\
&=\|a_\parallel\|^2+\|a_\perp\|^2
+2a_\parallel^\top a_\perp
\end{aligned}
\]

因为交叉项为零：

\[
\boxed{
\|a\|^2
=
\|\Pi a\|^2
+
\|\Pi_\perp a\|^2
}
\tag{C.4.4}
\]

**你之前将它理解成直角三角形的斜边分解，是完全正确的。**

需要注意的是，在高维空间中，两个正交分量不一定分别沿某条原始坐标轴，它们可以位于任意倾斜的子空间中。

---

## C.5 任意倾斜子空间的坐标变换

前面的公式：

\[
\Pi=PP^\top
\]

并不要求子空间与原始坐标轴对齐。

为了证明这一点，我们构造一组完整的标准正交基。

### C.5.1 补全子空间的基

已知：

\[
P\in\mathbb R^{D\times d}
\]

其列向量构成 \(\mathcal S\) 的标准正交基。

现在选择：

\[
P_\perp\in\mathbb R^{D\times(D-d)}
\]

使它的列向量构成 \(\mathcal S^\perp\) 的标准正交基。

它们满足：

\[
\boxed{
\begin{aligned}
P^\top P&=I_d\\
P_\perp^\top P_\perp&=I_{D-d}\\
P^\top P_\perp&=0
\end{aligned}
}
\tag{C.5.1}
\]

将两组基拼接：

\[
\boxed{
Q=[P,P_\perp]\in\mathbb R^{D\times D}
}
\tag{C.5.2}
\]

由于所有列向量构成 \(\mathbb R^D\) 的完整标准正交基：

\[
\boxed{
Q^\top Q=QQ^\top=I_D
}
\tag{C.5.3}
\]

所以 \(Q\) 是一个正交矩阵。

由：

\[
QQ^\top=I_D
\]

还可以得到：

\[
\boxed{
PP^\top+P_\perp P_\perp^\top=I_D
}
\tag{C.5.4}
\]

因此：

\[
\boxed{
I_D-\Pi=P_\perp P_\perp^\top
}
\tag{C.5.5}
\]

这说明：

- \(PP^\top\)：投影到数据子空间。
- \(P_\perp P_\perp^\top\)：投影到正交补空间。

### C.5.2 新坐标系下的投影

对于任意向量：

\[
a\in\mathbb R^D
\]

定义它在新基 \(Q\) 下的坐标：

\[
\boxed{
z=Q^\top a
}
\tag{C.5.6}
\]

由于：

\[
Q^\top=
\begin{bmatrix}
P^\top\\
P_\perp^\top
\end{bmatrix}
\]

所以：

\[
z=
\begin{bmatrix}
P^\top a\\
P_\perp^\top a
\end{bmatrix}
\]

记：

\[
z_\parallel=P^\top a\in\mathbb R^d
\]

\[
z_\perp=P_\perp^\top a\in\mathbb R^{D-d}
\]

于是：

\[
\boxed{
z=
\begin{bmatrix}
z_\parallel\\
z_\perp
\end{bmatrix}
}
\tag{5.7}
\]

其中 \(z_\parallel\) 是子空间内的坐标，\(z_\perp\) 是正交补空间的坐标。

现在计算投影算子在新坐标系下的矩阵表示：

\[
[\Pi]_Q=Q^\top\Pi Q
\]

代入：

\[
\Pi=PP^\top
\]

得到：

\[
[\Pi]_Q
=
\begin{bmatrix}
P^\top\\
P_\perp^\top
\end{bmatrix}
PP^\top
\begin{bmatrix}
P&P_\perp
\end{bmatrix}
\]

利用式（C.5.1）的正交关系，得到：

\[
\boxed{
[\Pi]_Q=
\begin{bmatrix}
I_d&0\\
0&0_{D-d}
\end{bmatrix}
}
\tag{5.8}
\]

同理：

\[
\boxed{
[\Pi_\perp]_Q=
\begin{bmatrix}
0_d&0\\
0&I_{D-d}
\end{bmatrix}
}
\tag{5.9}
\]

**这是整个投影推导最直观的结果。**

即使目标子空间在原始坐标系中任意倾斜，我们总能找到一组与它对齐的标准正交坐标系。

在这组坐标系中，投影只是：

\[
\begin{bmatrix}
z_\parallel\\z_\perp
\end{bmatrix}
\longrightarrow
\begin{bmatrix}
z_\parallel\\0
\end{bmatrix}
\]

而正交补投影是：

\[
\begin{bmatrix}
z_\parallel\\z_\perp
\end{bmatrix}
\longrightarrow
\begin{bmatrix}
0\\z_\perp
\end{bmatrix}
\]

因此，**子空间维度指的是独立坐标的数量，而不是原始坐标中非零元素的数量。**

---

## C.6 高斯噪声的平均平方能量

这是后面能量推导的概率论基础。

### C.6.1 标准高斯噪声

设：

\[
\boxed{
x_0\sim\mathcal N(0,I_D)
}
\tag{C.6.1}
\]

即：

\[
x_0=
\begin{bmatrix}
\epsilon_1\\
\epsilon_2\\
\vdots\\
\epsilon_D
\end{bmatrix}
\]

其中：

\[
\epsilon_i\overset{\mathrm{i.i.d.}}{\sim}\mathcal N(0,1)
\]

对于每个分量：

\[
\mathbb E[\epsilon_i]=0
\]

\[
\operatorname{Var}(\epsilon_i)=1
\]

根据方差定义：

\[
\operatorname{Var}(\epsilon_i)
=
\mathbb E[\epsilon_i^2]
-
\big(\mathbb E[\epsilon_i]\big)^2
\]

得到：

\[
\boxed{
\mathbb E[\epsilon_i^2]=1
}
\tag{C.6.2}
\]

### C.6.2 总平方能量

定义向量的平方能量：

\[
E(x_0)=\|x_0\|^2
\]

则：

\[
\|x_0\|^2
=
\sum_{i=1}^D\epsilon_i^2
\]

取期望：

\[
\begin{aligned}
\mathbb E\|x_0\|^2
&=\mathbb E\left[\sum_{i=1}^D\epsilon_i^2\right]\\
&=\sum_{i=1}^D\mathbb E[\epsilon_i^2]\\
&=\sum_{i=1}^D1
\end{aligned}
\]

所以：

\[
\boxed{
\mathbb E\|x_0\|^2=D
}
\tag{C.6.3}
\]

这里的“能量”指平方欧氏范数，并不是物理学中具有特定单位的能量。

还要注意：

\[
\mathbb E\|x_0\|^2=D
\]

不意味着每次采样都满足：

\[
\|x_0\|^2=D
\]

实际上：

\[
\|x_0\|^2\sim\chi_D^2
\]

即自由度为 \(D\) 的卡方分布。

它的期望为：

\[
D
\]

方差为：

\[
2D
\]

因此这里讨论的是**平均平方能量**。

---

## C.7 标准高斯分布的旋转不变性

前面我们已经知道：

\[
x_0\sim\mathcal N(0,I_D)
\]

但目标子空间可能是任意倾斜的。

那么，换到与子空间对齐的坐标系后，噪声的统计性质会不会发生变化？

答案是不会。

### C.7.1 对噪声进行正交坐标变换

使用附录 C.5 构造的正交矩阵：

\[
Q=[P,P_\perp]
\]

定义：

\[
\boxed{
z_0=Q^\top x_0
}
\tag{C.7.1}
\]

由于 \(x_0\) 是高斯随机向量，线性变换后的 \(z_0\) 仍然是高斯随机向量。

计算均值：

\[
\begin{aligned}
\mathbb E[z_0]
&=\mathbb E[Q^\top x_0]\\
&=Q^\top\mathbb E[x_0]\\
&=0
\end{aligned}
\]

计算协方差：

\[
\operatorname{Cov}(z_0)
=
Q^\top\operatorname{Cov}(x_0)Q
\]

由于：

\[
\operatorname{Cov}(x_0)=I_D
\]

所以：

\[
\operatorname{Cov}(z_0)=Q^\top I_DQ
\]

又因为：

\[
Q^\top Q=I_D
\]

得到：

\[
\boxed{
\operatorname{Cov}(z_0)=I_D
}
\]

因此：

\[
\boxed{
z_0=Q^\top x_0\sim\mathcal N(0,I_D)
}
\tag{C.7.2}
\]

这就是标准高斯分布的旋转不变性。

它说明：

> 无论选择哪一组标准正交坐标轴，标准高斯噪声在每个新坐标方向上的方差仍然为1。

这也是为什么倾斜子空间不会改变最终的能量计算。

---

## C.8 二次型期望、矩阵迹与投影维数


### C.8.1 二次型期望与协方差

现在不用显式构造新坐标系，直接在原始坐标中推导。

首先介绍一个恒等式。

设随机向量：

\[
X\in\mathbb R^D
\]

满足：

\[
\mathbb E[X]=0
\]

协方差矩阵：

\[
\Sigma=\mathbb E[XX^\top]
\]

那么，对于任意 \(D\times D\) 矩阵 \(A\)：

\[
\boxed{
\mathbb E[X^\top AX]=\operatorname{tr}(A\Sigma)
}
\tag{C.8.1}
\]

**证明：**

由于 \(X^\top AX\) 是一个标量：

\[
X^\top AX=\operatorname{tr}(X^\top AX)
\]

利用迹的循环性质：

\[
\operatorname{tr}(X^\top AX)
=
\operatorname{tr}(AXX^\top)
\]

取期望：

\[
\begin{aligned}
\mathbb E[X^\top AX]
&=\mathbb E[\operatorname{tr}(AXX^\top)]\\
&=\operatorname{tr}\left(A\mathbb E[XX^\top]\right)\\
&=\operatorname{tr}(A\Sigma)
\end{aligned}
\]

证毕。

对于标准高斯：

\[
x_0\sim\mathcal N(0,I_D)
\]

有：

\[
\Sigma=I_D
\]

所以：

\[
\boxed{
\mathbb E[x_0^\top Ax_0]=\operatorname{tr}(A)
}
\tag{C.8.2}
\]

现在计算：

\[
\mathbb E\|\Pi_\perp x_0\|^2
\]

根据范数定义：

\[
\begin{aligned}
\|\Pi_\perp x_0\|^2
&=(\Pi_\perp x_0)^\top(\Pi_\perp x_0)\\
&=x_0^\top\Pi_\perp^\top\Pi_\perp x_0
\end{aligned}
\]

由于 \(\Pi_\perp\) 对称且幂等：

\[
\Pi_\perp^\top\Pi_\perp
=
\Pi_\perp^2
=
\Pi_\perp
\]

因此：

\[
\|\Pi_\perp x_0\|^2
=
x_0^\top\Pi_\perp x_0
\]

取期望：

\[
\mathbb E\|\Pi_\perp x_0\|^2
=
\mathbb E[x_0^\top\Pi_\perp x_0]
\]

根据上面的二次型期望公式（C.8.2）：

\[
\boxed{
\mathbb E\|\Pi_\perp x_0\|^2
=
\operatorname{tr}(\Pi_\perp)
}
\tag{C.8.3}
\]

接下来：

\[
\Pi_\perp=I_D-PP^\top
\]

所以：

\[
\begin{aligned}
\operatorname{tr}(\Pi_\perp)
&=\operatorname{tr}(I_D-PP^\top)\\
&=\operatorname{tr}(I_D)-\operatorname{tr}(PP^\top)
\end{aligned}
\]

第一项：

\[
\operatorname{tr}(I_D)=D
\]

第二项，利用迹的循环性质：

\[
\operatorname{tr}(PP^\top)
=
\operatorname{tr}(P^\top P)
\]

因为：

\[
P^\top P=I_d
\]

所以：

\[
\operatorname{tr}(PP^\top)=d
\]

最终：

\[
\boxed{
\mathbb E\|\Pi_\perp x_0\|^2=D-d
}
\tag{C.8.4}
\]

两种方法得到完全相同的结果。

### C.8.2 为什么投影矩阵的迹等于子空间维度？

除了上面的计算，还可以从特征值理解。

正交投影矩阵满足：

\[
\Pi^2=\Pi
\]

假设：

\[
\Pi q=\lambda q
\]

其中 \(q\neq0\) 是特征向量。

那么：

\[
\Pi^2q=\lambda^2q
\]

但：

\[
\Pi^2q=\Pi q=\lambda q
\]

所以：

\[
\lambda^2=\lambda
\]

得到：

\[
\lambda(\lambda-1)=0
\]

因此：

\[
\boxed{
\lambda\in\{0,1\}
}
\tag{C.8.5}
\]

对于投影到 \(d\) 维子空间的矩阵 \(\Pi\)：

- 子空间内有 \(d\) 个独立方向，对应特征值1。
- 正交补中有 \(D-d\) 个独立方向，对应特征值0。

由于矩阵的迹等于特征值之和：

\[
\operatorname{tr}(\Pi)
=
\underbrace{1+\cdots+1}_{d\text{个}}
+
\underbrace{0+\cdots+0}_{D-d\text{个}}
\]

所以：

\[
\boxed{
\operatorname{tr}(\Pi)=d
}
\tag{C.8.6}
\]

同理：

\[
\boxed{
\operatorname{tr}(\Pi_\perp)=D-d
}
\tag{C.8.7}
\]

因此，可以将前面的结果理解为：

> 标准高斯噪声在任意一个 \(k\) 维线性子空间中的平均平方投影长度，恰好等于 \(k\)。

---

## C.9 从附录结论回到正文

正文 3.3 节只需要组合下面三条结果：

1. 若 $x_1\in\mathcal S$，则 $\Pi_\perp x_1=0$。
2. 若 $x_0$ 零均值且协方差为 $I_D$，则 $\mathbb E\|\Pi_\perp x_0\|^2=\operatorname{tr}(\Pi_\perp)=D-d$。
3. 对条件速度 $v=x_1-x_0$，有 $\Pi_\perp v=-\Pi_\perp x_0$。

因此立即得到：

\[
\boxed{
\mathbb E\|\Pi_\perp v\|^2=D-d
}
\]

这条结论适用于任意朝向的 $d$ 维线性数据子空间。
